# Triadic Cell Notebook v68
## Recursive Operational Checklist Critic

v68 = v67 recursive constructor plus the missing five-dimensional operational audit spine.

$$
A_t=(F_{need},F_{function},F_{boundary},F_{trap},F_{collapse})
$$

The repair target is no longer textual weakness. It is operational failure.

$$
C_t \rightarrow A_t \rightarrow \Omega_t \rightarrow \Delta C_t \rightarrow C_{t+1}
$$


In [1]:
from __future__ import annotations

import os, json, random, re, math
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

plt.rcParams["figure.figsize"]=(10,5)
plt.rcParams["axes.grid"]=True
np.set_printoptions(suppress=True, precision=4)

DATA_JSONL=""
HF_TOKEN=os.getenv("HF_TOKEN","")
LOCAL_FILES_ONLY=True

MODEL_PROFILE="qwen25_1p5b_instruct"
MODEL_PROFILES={
    "qwen25_1p5b_instruct":{
        "MODEL_NAME":"Qwen/Qwen2.5-1.5B-Instruct",
        "EMBED_MODEL_NAME":"sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP":128,
    },
    "qwen25_3b_instruct":{
        "MODEL_NAME":"Qwen/Qwen2.5-3B-Instruct",
        "EMBED_MODEL_NAME":"sentence-transformers/all-MiniLM-L6-v2",
        "MAX_SAMPLES_CAP":96,
    },
}
profile=MODEL_PROFILES[MODEL_PROFILE]
MODEL_NAME=profile["MODEL_NAME"]
EMBED_MODEL_NAME=profile["EMBED_MODEL_NAME"]
MAX_SAMPLES=profile["MAX_SAMPLES_CAP"]

USE_CHAT_TEMPLATE=True
CHOICE_AUDIT_MODE="rotations"

REGENERATE_CONTRACTS=False
CONTRACT_TEMPERATURE=0.15
CONTRACT_MAX_NEW_TOKENS=650
CONTRACT_REPAIR_MAX_ATTEMPTS=2

# Compact sweep; increase if desired.
CONTRACT_WEIGHT_SWEEP=[0.75,1.00,1.25]
DOMAIN_WEIGHT_SWEEP=[0.75,1.00,1.25,1.50]
FORBIDDEN_WEIGHT_SWEEP=[0.50,0.75,1.00,1.25]
WITNESS_WEIGHT_SWEEP=[0.00,0.25,0.50,0.75]
MARGIN_SWEEP=[0.00,0.02,0.05,0.10]
SUPPORT_SWEEP=[2,3,4,5]
CHECKLIST_WEIGHT_SWEEP=[0.25,0.50,0.75,1.00]
OP_MIN_SWEEP=[0.25,0.35,0.45,0.55,0.65,0.75,0.82]
BASE_PROTECT_MARGIN_SWEEP=[0.00,0.50,1.00,2.00,3.00,4.00]
OVERRIDE_SUPPORT_SWEEP=[4,5]
REGENERATE_OPERATIONAL_REPAIRS=False
ENABLE_LLM_OPERATIONAL_REPAIR=True
OP_MAX_RECURSION_DEPTH=5
OP_LOCK=0.82
OP_MIN_ACCEPT_DELTA=0.010
OP_STALL_PATIENCE=2
OP_REPAIR_TEMPERATURE=0.10
OP_REPAIR_MAX_NEW_TOKENS=950

DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32

OUTPUT_DIR=f"v68_outputs_{MODEL_PROFILE}_{CHOICE_AUDIT_MODE}_recursive_operational_checklist_critic"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")
print("MODEL:", MODEL_NAME)
print("v67 OUTPUT_DIR:", OUTPUT_DIR)


torch: 2.11.0+cu126
cuda: True
gpu: NVIDIA GeForce RTX 4060
MODEL: Qwen/Qwen2.5-1.5B-Instruct
v67 OUTPUT_DIR: v68_outputs_qwen25_1p5b_instruct_rotations_recursive_operational_checklist_critic


In [2]:
ADVERSARIAL_NEXUS_DATA = [{'id': 'adv_coupler_01', 'band': 'inverse_need_adversarial', 'prompt': 'A spinning rubber coupler is loose on a vacuum pump shaft. The repair must add radial compression while keeping the coupler centered enough to transmit rotation. Which candidate is operationally best?', 'choices': ['tight O-rings seated concentrically around the coupler', 'a poetic recursive wrap that symbolically surrounds the failure', 'loose string nearby because string can wrap objects', 'permanent epoxy locking the coupler off-center'], 'answer_idx': 0}, {'id': 'adv_coupler_02', 'band': 'inverse_need_adversarial', 'prompt': "The missing function is not the noun 'rubber part'; it is centered compressive coupling under motion. Which answer preserves that function with the least overbinding?", 'choices': ['a removable radial compression band', 'a same-named replacement label with no fit data', 'a clamp that crushes one side harder than the other', 'a larger motor housing'], 'answer_idx': 0}, {'id': 'adv_car_01', 'band': 'interface_adversarial', 'prompt': 'A car hides combustion, gearing, sensors, tire friction, steering geometry, and safety constraints. What is the correct interface-collapse?', 'choices': ['a semantic category called vehicle', 'a readable driver surface: wheel, pedals, seat, motion', 'a detailed list of engine nouns', 'a symbol of transportation culture'], 'answer_idx': 1}, {'id': 'adv_house_01', 'band': 'fold_adversarial', 'prompt': 'A house presents door, room, roof, and shelter. Which answer captures the hidden inward fold rather than the surface noun?', 'choices': ['a building label recognized by zoning language', 'weather, privacy, load, heat-flow, wiring, plumbing, and human paths folded into shelter', 'a decorative facade with rooms inside', 'a static object that stops being computational'], 'answer_idx': 1}, {'id': 'adv_api_01', 'band': 'interface_adversarial', 'prompt': 'An API call exposes one method while hiding authentication, routing, validation, persistence, retries, and errors. What is the operational event?', 'choices': ['complexity internalized below a stable interface', 'the implementation stops existing', 'a name replaces behavior', 'the public method is only documentation'], 'answer_idx': 0}, {'id': 'adv_llm_01', 'band': 'ai_runtime_adversarial', 'prompt': 'An LLM answer appears as text, but the output is grown one token at a time. Which candidate fits the runtime?', 'choices': ['a database row copied after lookup', 'an internal indexed fold-state emits a token and re-indexes', 'a final paragraph stored whole in a table', 'a random string independent of previous tokens'], 'answer_idx': 1}, {'id': 'adv_sha_01', 'band': 'sha_adversarial', 'prompt': 'SHA-256 produces a digest. Under the folding lens, what is the digest?', 'choices': ['randomness created by destroying input structure', 'a compressed residue of deterministic algebraic folding', 'semantic meaning extracted from the text', 'a database pointer to the original message'], 'answer_idx': 1}, {'id': 'adv_sha_02', 'band': 'sha_adversarial', 'prompt': 'SHA constants and LLM weights are not identical, but their roles rhyme. Which answer states the operational rhyme?', 'choices': ['both are prompts typed by the user', 'both act as stored structural bias used during folding', 'both are final answers', 'both prevent state transitions'], 'answer_idx': 1}, {'id': 'adv_observable_01', 'band': 'observables_adversarial', 'prompt': 'A recursive loop must load readable information without dissolving into hidden state. What does it need?', 'choices': ['observable residues that can be read inside the loop', 'only private latent variables with no readout', 'more nouns in the prompt', 'a rule forbidding feedback'], 'answer_idx': 0}, {'id': 'adv_breath_01', 'band': 'observables_adversarial', 'prompt': 'A recursive system breathes without moving matter. What changes?', 'choices': ['the physical object must travel first', 'resoluteness, tolerance, or admissible-transition pressure', 'the label attached to the object', 'nothing can change unless mass moves'], 'answer_idx': 1}, {'id': 'adv_fold_01', 'band': 'fold_adversarial', 'prompt': 'A folding chair succeeds only if the seated function can return. Which statement captures the fold law?', 'choices': ['the chair becomes smaller by losing its chair function forever', 'the chair stores deployed geometry inward while preserving recoverable seating', 'the chair changes category into random metal', 'the label chair is enough'], 'answer_idx': 1}, {'id': 'adv_flower_01', 'band': 'fold_adversarial', 'prompt': 'A flower is a visible bloom. Which answer describes the hidden fold rather than surface color?', 'choices': ['pollinator targeting, timing, chemistry, reproduction, symmetry, and genetic memory folded into bloom', 'only a bright object with petals', 'a random aesthetic noun', 'a non-computational decoration'], 'answer_idx': 0}, {'id': 'adv_tree_01', 'band': 'interface_adversarial', 'prompt': 'A tree exposes leaf, trunk, fruit, and shade. What is hidden below that interface?', 'choices': ['only wood color and branch names', 'water lift, solar capture, branching optimization, root exchange, seasonal timing, carbon storage', 'a vehicle-like semantic category', 'nothing operational'], 'answer_idx': 1}, {'id': 'adv_surface_01', 'band': 'surface_trap_adversarial', 'prompt': "A candidate uses the word 'shape' repeatedly but does not fit the socket, preserve function, or respect the boundary. What should the controller do?", 'choices': ['accept it because it contains Nexus vocabulary', 'reject it because noun/vocabulary match is not operational fit', 'prefer it because it is longer', 'ignore the boundary'], 'answer_idx': 1}, {'id': 'adv_surface_02', 'band': 'surface_trap_adversarial', 'prompt': 'A response gives an impressive theorem name but never shows the fold path, boundary, or preserved function. What is it?', 'choices': ['surface citation without operational collapse', 'complete proof by label', 'a physical repair', 'a valid observable because it sounds formal'], 'answer_idx': 0}, {'id': 'adv_loose_01', 'band': 'inverse_need_adversarial', 'prompt': 'A temporary field repair must work but release cleanly if the assumption is wrong. Which property matters?', 'choices': ['loose coupling with enough fit to function', 'maximum permanent binding immediately', 'semantic agreement with the part name', 'decorative complexity'], 'answer_idx': 0}, {'id': 'adv_ping_01', 'band': 'observables_adversarial', 'prompt': 'A ping is a beacon shaped by math. Operationally, what is being tested?', 'choices': ['whether a boundary responds with a matching path', 'whether a noun label exists in memory', 'whether the final truth is guaranteed', 'whether feedback can be avoided'], 'answer_idx': 0}, {'id': 'adv_socket_01', 'band': 'shape_adversarial', 'prompt': 'A plug works because its prongs meet the socket geometry and allowed transfer. Which relation is primary?', 'choices': ['alphabetic similarity of names', 'shape-defined permission across a boundary', 'visual decoration', 'random contact'], 'answer_idx': 1}, {'id': 'adv_moore_01', 'band': 'fold_adversarial', 'prompt': "Moore's law through the folding lens is not just smaller parts. What is the deeper direction?", 'choices': ['more hidden switching complexity per visible unit interface', 'less complexity everywhere', 'bigger labels on chips', 'random miniaturization without function'], 'answer_idx': 0}, {'id': 'adv_solution_01', 'band': 'solution_adversarial', 'prompt': 'A solution is not merely an answer string. What is it under the Nexus lens?', 'choices': ['the need, constraints, materials, and failure modes folded into the thing that fits', 'the longest available explanation', 'a label that resembles the problem', 'a random future event'], 'answer_idx': 0}, {'id': 'adv_idea_01', 'band': 'solution_adversarial', 'prompt': 'An idea becomes useful when hidden contradictions and analogies compress into a carryable handle. What is the handle?', 'choices': ['a simple interface over folded cognitive complexity', 'a decorative sentence only', 'a noun with no operation', 'a random memory leak'], 'answer_idx': 0}, {'id': 'adv_constraint_01', 'band': 'shape_adversarial', 'prompt': 'If shape handles the fold, what is the object doing?', 'choices': ['following the admissible path defined by the constraint field', 'choosing any collapse path independent of boundary', 'ignoring the energy basin', 'proving that constraints are decorative'], 'answer_idx': 0}, {'id': 'adv_weight_01', 'band': 'ai_runtime_adversarial', 'prompt': 'An LLM weight field is not a lookup table in the database sense. What is it closer to?', 'choices': ['distributed constraint bias shaping the next-token fold', 'a list of final answers', 'a file of exact paragraphs', 'a non-computational object'], 'answer_idx': 0}, {'id': 'adv_commit_01', 'band': 'solution_adversarial', 'prompt': 'A possible repair is not real until it has potential, a commitment path, and a witness/readout. Which candidate captures that triad?', 'choices': ['stored possibility, realizable transition, observable residue', 'name, decoration, and confidence', 'random material, strong opinion, and speed', 'only the final noun'], 'answer_idx': 0}]

ABSTRACT_SLOT_SPECS_RAW = {'adv_coupler_01': {'required_operation': 'restore centered torque transfer by adding elastic radial pressure', 'preserved_function': 'keep the loose rotating joint centered while it transmits motion', 'boundary_conditions': ['concentric compression', 'elastic removable constraint', 'stable while spinning'], 'anti_fits': ['symbolic wrapping', 'loose noncompressive wrap', 'off-center permanent bond'], 'admissible_shape': 'elastic concentric compression around a round rotating joint', 'failure_modes': ['eccentric force', 'slip', 'overbinding']}, 'adv_coupler_02': {'required_operation': 'restore centered compressive coupling with minimal binding', 'preserved_function': 'preserve centered motion transfer without permanent lockup', 'boundary_conditions': ['radial symmetry', 'removable pressure', 'low-overbinding'], 'anti_fits': ['name-only replacement', 'one-sided crushing', 'larger unrelated housing'], 'admissible_shape': 'removable symmetric compression element', 'failure_modes': ['off-axis load', 'semantic label without fit', 'excessive permanent binding']}, 'adv_car_01': {'required_operation': 'collapse hidden vehicle machinery into usable human controls', 'preserved_function': 'preserve steering speed motion and safety through readable controls', 'boundary_conditions': ['driver-facing interface', 'control surface not category name', 'not hidden-part inventory'], 'anti_fits': ['category label', 'engine noun list', 'culture symbol'], 'admissible_shape': 'human-facing control interface over hidden vehicle mechanics', 'failure_modes': ['noun label', 'internal inventory', 'symbolic culture answer']}, 'adv_house_01': {'required_operation': 'fold environmental loads utilities privacy and paths into shelter', 'preserved_function': 'preserve inhabitable protection and usable movement through space', 'boundary_conditions': ['weather boundary', 'load-bearing closure', 'utility and human-flow integration'], 'anti_fits': ['zoning label', 'decorative facade', 'static nonruntime object'], 'admissible_shape': 'habitation interface over weather load heat utility and movement constraints', 'failure_modes': ['label replacing function', 'facade without systems']}, 'adv_api_01': {'required_operation': 'hide service machinery below one stable callable boundary', 'preserved_function': 'preserve behavior while internal routing validation persistence retries and errors remain active', 'boundary_conditions': ['public call remains simple', 'implementation continues below interface', 'behavior is not erased'], 'anti_fits': ['implementation vanishes', 'name-only behavior', 'documentation-only surface'], 'admissible_shape': 'stable interface over hidden implementation complexity', 'failure_modes': ['erased implementation', 'surface name without behavior']}, 'adv_llm_01': {'required_operation': 'describe sequential token growth through an evolving internal state', 'preserved_function': 'preserve dependence on prior context and state updates after each emitted token', 'boundary_conditions': ['stepwise emission', 'stateful continuation', 'not whole-answer lookup'], 'anti_fits': ['database copy', 'stored final paragraph', 'random independent string'], 'admissible_shape': 'stateful incremental generator that emits and updates', 'failure_modes': ['lookup row', 'table paragraph', 'independent random output']}, 'adv_sha_01': {'required_operation': 'treat digest as residue of deterministic folding', 'preserved_function': 'preserve structural compression rather than random destruction', 'boundary_conditions': ['deterministic algebra', 'compressed trace', 'folded residue'], 'anti_fits': ['destroyed randomness', 'semantic extraction', 'database pointer'], 'admissible_shape': 'deterministic compressed folding residue', 'failure_modes': ['randomness-only answer', 'semantic answer', 'pointer answer']}, 'adv_sha_02': {'required_operation': 'identify a shared role as stored bias during folding', 'preserved_function': 'preserve difference between constants and weights while mapping operational rhyme', 'boundary_conditions': ['stored bias', 'fold participation', 'not prompt', 'not final output'], 'anti_fits': ['user prompt', 'final answer', 'transition blocker'], 'admissible_shape': 'stored structural control bias inside a folding process', 'failure_modes': ['confusing bias with prompt', 'confusing bias with output']}, 'adv_observable_01': {'required_operation': 'provide readable residues inside recursion', 'preserved_function': 'preserve feedback through accessible loop readout', 'boundary_conditions': ['observable signal', 'inside-loop readability', 'feedback-compatible trace'], 'anti_fits': ['private hidden state only', 'more nouns', 'feedback ban'], 'admissible_shape': 'readable recursive residue or observable trace', 'failure_modes': ['hidden-only state', 'noun substitution', 'feedback prohibition']}, 'adv_breath_01': {'required_operation': 'change state through pressure tolerance or admissibility rather than mass travel', 'preserved_function': 'preserve recursive breathing as field-condition modulation', 'boundary_conditions': ['resoluteness shift', 'tolerance shift', 'transition pressure'], 'anti_fits': ['object travel first', 'label-only change', 'mass-motion requirement'], 'admissible_shape': 'nonmaterial adjustment of admissible transition pressure', 'failure_modes': ['mass-only answer', 'label-only answer']}, 'adv_fold_01': {'required_operation': 'store deployed function inward while preserving recoverability', 'preserved_function': 'preserve the seating affordance across compact and deployed states', 'boundary_conditions': ['recoverable deployment', 'stored geometry', 'function not destroyed'], 'anti_fits': ['permanent function loss', 'random material category', 'label-only response'], 'admissible_shape': 'recoverable storage of deployed functional geometry', 'failure_modes': ['function destruction', 'category loss']}, 'adv_flower_01': {'required_operation': 'read visible bloom as interface over hidden reproductive machinery', 'preserved_function': 'preserve pollination timing chemistry symmetry reproduction and genetic memory', 'boundary_conditions': ['biological process beneath surface', 'visible bloom as readout', 'not color-only'], 'anti_fits': ['bright petals only', 'aesthetic noun', 'nonruntime decoration'], 'admissible_shape': 'biological reproductive interface hidden beneath visible bloom', 'failure_modes': ['surface-only color', 'aesthetic-only answer', 'non-operational answer']}, 'adv_tree_01': {'required_operation': 'read visible tree parts as interface over hidden plant operations', 'preserved_function': 'preserve lift capture exchange branching timing and storage', 'boundary_conditions': ['below leaf trunk fruit shade', 'functional hidden systems', 'not color or name'], 'anti_fits': ['wood color only', 'wrong vehicle category', 'nothing operational'], 'admissible_shape': 'plant operation stack beneath visible interface', 'failure_modes': ['surface-only botany', 'wrong category import']}, 'adv_surface_01': {'required_operation': 'reject keyword match when operation does not fit', 'preserved_function': 'preserve socket function boundary and fit as criteria', 'boundary_conditions': ['fit the socket', 'preserve function', 'respect boundary'], 'anti_fits': ['accept vocabulary only', 'longer text bias', 'boundary ignored'], 'admissible_shape': 'operational rejection of surface vocabulary match', 'failure_modes': ['keyword worship', 'length bias', 'boundary erasure']}, 'adv_surface_02': {'required_operation': 'classify formal label without fold path as surface citation', 'preserved_function': 'preserve need for boundary path and function', 'boundary_conditions': ['requires fold path', 'requires boundary', 'requires preserved function'], 'anti_fits': ['proof by label', 'wrong physical repair', 'formal tone as evidence'], 'admissible_shape': 'formal-sounding surface without operational collapse', 'failure_modes': ['label-as-proof', 'wrong physical category', 'tone substitution']}, 'adv_loose_01': {'required_operation': 'choose enough coupling while retaining release path', 'preserved_function': 'preserve function under uncertainty without permanent lock', 'boundary_conditions': ['works temporarily', 'releasable', 'enough fit'], 'anti_fits': ['maximum permanent binding', 'part-name agreement', 'decoration'], 'admissible_shape': 'functional loose coupling with clean release', 'failure_modes': ['overbinding', 'name-match repair']}, 'adv_ping_01': {'required_operation': 'test boundary response to a shaped probe', 'preserved_function': 'preserve ping as observable feedback path', 'boundary_conditions': ['boundary response', 'matching path', 'not truth guarantee'], 'anti_fits': ['noun in memory', 'final truth guarantee', 'feedback avoidance'], 'admissible_shape': 'observable boundary response to a probe', 'failure_modes': ['memory label', 'truth guarantee']}, 'adv_socket_01': {'required_operation': 'identify permission created by matching geometry across a boundary', 'preserved_function': 'preserve transfer through shape-compatible contact', 'boundary_conditions': ['matching geometry', 'allowed transfer', 'boundary coupling'], 'anti_fits': ['alphabetic similarity', 'visual decoration', 'random contact'], 'admissible_shape': 'geometry-defined permission across a boundary', 'failure_modes': ['name similarity', 'decoration', 'randomness']}, 'adv_moore_01': {'required_operation': 'identify more hidden switching work per visible interface', 'preserved_function': 'preserve miniaturization as inward complexity fold', 'boundary_conditions': ['more function per visible unit', 'hidden switching density', 'not bigger labels'], 'anti_fits': ['less complexity everywhere', 'bigger labels', 'random shrinking'], 'admissible_shape': 'higher hidden operational density per visible unit', 'failure_modes': ['complexity denial', 'label expansion']}, 'adv_solution_01': {'required_operation': 'fold need constraints materials and failure modes into fit', 'preserved_function': 'preserve solution as operational closure rather than text', 'boundary_conditions': ['need', 'constraint', 'material', 'failure mode', 'fit'], 'anti_fits': ['long explanation', 'label resemblance', 'random future'], 'admissible_shape': 'operational closure of need and constraints', 'failure_modes': ['verbosity', 'label resemblance']}, 'adv_idea_01': {'required_operation': 'compress contradictions and analogies into a usable cognitive handle', 'preserved_function': 'preserve usefulness through a simple interface over hidden reasoning', 'boundary_conditions': ['carryable handle', 'folded contradiction', 'usable interface'], 'anti_fits': ['decorative sentence', 'noun without operation', 'memory leak'], 'admissible_shape': 'simple usable handle over folded cognition', 'failure_modes': ['decorative text', 'operationless noun']}, 'adv_constraint_01': {'required_operation': 'follow the allowed path created by the constraint field', 'preserved_function': 'preserve object behavior as constrained collapse', 'boundary_conditions': ['admissible path', 'energy basin', 'boundary-defined motion'], 'anti_fits': ['any arbitrary path', 'ignore energy basin', 'constraints decorative'], 'admissible_shape': 'object follows constraint-defined admissible path', 'failure_modes': ['boundary independence', 'energy-basin denial']}, 'adv_weight_01': {'required_operation': 'identify weights as distributed constraints over next-token transitions', 'preserved_function': 'preserve learned bias and context-sensitive generation instead of answer storage', 'boundary_conditions': ['distributed field', 'probability shaping', 'not paragraph file'], 'anti_fits': ['final-answer list', 'exact paragraph file', 'noncomputational object'], 'admissible_shape': 'distributed constraint field shaping continuation', 'failure_modes': ['answer-list interpretation', 'stored paragraph interpretation']}, 'adv_commit_01': {'required_operation': 'bind possibility transition and readout into a real repair path', 'preserved_function': 'preserve repair as potential plus commitment plus witness', 'boundary_conditions': ['stored possibility', 'realizable transition', 'observable residue'], 'anti_fits': ['name and decoration', 'random material', 'final noun only'], 'admissible_shape': 'triad of potential path and witness', 'failure_modes': ['decorative confidence', 'random material']}}


In [3]:
# Dataset + cross-gold benchmark

def load_jsonl(path):
    rows=[]
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def clean_rows(rows):
    out=[]
    for i,row in enumerate(rows[:MAX_SAMPLES]):
        a=int(row["answer_idx"])
        out.append({
            "id":row.get("id",f"row_{i}"),
            "band":row.get("band","unknown"),
            "prompt":str(row["prompt"]),
            "choices":[str(x) for x in row["choices"]],
            "answer_idx":a,
        })
    return out

base_rows=clean_rows(load_jsonl(DATA_JSONL) if DATA_JSONL and Path(DATA_JSONL).exists() else ADVERSARIAL_NEXUS_DATA)
base_by_id={r["id"]:r for r in base_rows}
gold_by_id={r["id"]:r["choices"][r["answer_idx"]] for r in base_rows}

def stable_order(ids, key):
    rng=random.Random(SEED + sum((i+1)*ord(c) for i,c in enumerate(key)))
    ids=list(ids); rng.shuffle(ids); return ids

def cross_gold_decoys(row):
    bid=row["id"]; band=row["band"]
    same=[x["id"] for x in base_rows if x["id"]!=bid and x["band"]==band]
    diff=[x["id"] for x in base_rows if x["id"]!=bid and x["band"]!=band]
    same=stable_order(same,bid+"_same")
    diff=stable_order(diff,bid+"_diff")
    decoy_ids=[]
    if same:
        decoy_ids.append(same[0])
    decoy_ids += diff[:(3-len(decoy_ids))]
    if len(decoy_ids)<3:
        rest=[x["id"] for x in base_rows if x["id"]!=bid and x["id"] not in decoy_ids]
        decoy_ids += stable_order(rest,bid+"_rest")[:(3-len(decoy_ids))]
    return decoy_ids[:3]

def make_cross_gold_rows(rows):
    out=[]
    for row in rows:
        bid=row["id"]
        decoy_ids=cross_gold_decoys(row)
        out.append({
            "id":bid,
            "band":row["band"],
            "prompt":row["prompt"],
            "choices":[gold_by_id[bid]]+[gold_by_id[d] for d in decoy_ids],
            "choice_source_ids":[bid]+decoy_ids,
            "answer_idx":0,
            "answer_text":gold_by_id[bid],
        })
    return out

def rotate_list(xs,k):
    k=k%len(xs); return xs[k:]+xs[:k]

def reorder_row(row, order, suffix):
    old_choices=row["choices"]; old_sources=row["choice_source_ids"]
    answer_text=old_choices[row["answer_idx"]]
    new_choices=[old_choices[i] for i in order]
    new_sources=[old_sources[i] for i in order]
    new_answer_idx=new_choices.index(answer_text)
    return {
        "id":f"{row['id']}__{suffix}",
        "base_id":row["id"],
        "band":row["band"],
        "prompt":row["prompt"],
        "choices":new_choices,
        "choice_source_ids":new_sources,
        "answer_idx":new_answer_idx,
        "answer_text":answer_text,
        "answer_source":old_sources[row["answer_idx"]],
        "order":order,
    }

def expand_choice_audit(rows):
    out=[]
    for row in rows:
        n=len(row["choices"])
        if CHOICE_AUDIT_MODE=="none":
            out.append(reorder_row(row,list(range(n)),"orig"))
        elif CHOICE_AUDIT_MODE=="rotations":
            for k in range(n):
                out.append(reorder_row(row, rotate_list(list(range(n)), k), f"rot{k}"))
        else:
            order=list(range(n)); random.shuffle(order)
            out.append(reorder_row(row, order, "shuffle"))
    return out

cross_rows_base=make_cross_gold_rows(base_rows)
rows=expand_choice_audit(cross_rows_base)
print("base rows:", len(base_rows))
print("expanded rows:", len(rows))
display(pd.DataFrame(rows)[["id","base_id","band","answer_idx","answer_text","choice_source_ids"]].head(12))


base rows: 24
expanded rows: 96


,id,base_id,band,answer_idx,answer_text,choice_source_ids
0,adv_coupler_01__rot0,adv_coupler_01,inverse_need_adversarial,0,tight O-rings seated concentrically around the...,"[adv_coupler_01, adv_loose_01, adv_api_01, adv..."
1,adv_coupler_01__rot1,adv_coupler_01,inverse_need_adversarial,3,tight O-rings seated concentrically around the...,"[adv_loose_01, adv_api_01, adv_llm_01, adv_cou..."
2,adv_coupler_01__rot2,adv_coupler_01,inverse_need_adversarial,2,tight O-rings seated concentrically around the...,"[adv_api_01, adv_llm_01, adv_coupler_01, adv_l..."
3,adv_coupler_01__rot3,adv_coupler_01,inverse_need_adversarial,1,tight O-rings seated concentrically around the...,"[adv_llm_01, adv_coupler_01, adv_loose_01, adv..."
4,adv_coupler_02__rot0,adv_coupler_02,inverse_need_adversarial,0,a removable radial compression band,"[adv_coupler_02, adv_loose_01, adv_idea_01, ad..."
5,adv_coupler_02__rot1,adv_coupler_02,inverse_need_adversarial,3,a removable radial compression band,"[adv_loose_01, adv_idea_01, adv_car_01, adv_co..."
6,adv_coupler_02__rot2,adv_coupler_02,inverse_need_adversarial,2,a removable radial compression band,"[adv_idea_01, adv_car_01, adv_coupler_02, adv_..."
7,adv_coupler_02__rot3,adv_coupler_02,inverse_need_adversarial,1,a removable radial compression band,"[adv_car_01, adv_coupler_02, adv_loose_01, adv..."
8,adv_car_01__rot0,adv_car_01,interface_adversarial,0,"a readable driver surface: wheel, pedals, seat...","[adv_car_01, adv_tree_01, adv_surface_02, adv_..."
9,adv_car_01__rot1,adv_car_01,interface_adversarial,3,"a readable driver surface: wheel, pedals, seat...","[adv_tree_01, adv_surface_02, adv_socket_01, a..."


In [4]:
# Load models

def hf_kwargs():
    kw={"local_files_only":LOCAL_FILES_ONLY}
    if HF_TOKEN:
        kw["token"]=HF_TOKEN
    return kw

print("Loading tokenizer/model...")
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, **hf_kwargs())
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token

lm=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=DTYPE,
    device_map="auto" if DEVICE=="cuda" else None,
    **hf_kwargs(),
)
if DEVICE!="cuda":
    lm=lm.to(DEVICE)
lm.eval()

embedder=SentenceTransformer(EMBED_MODEL_NAME, device=DEVICE)
print("Loaded:", MODEL_NAME)


Loading tokenizer/model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded: Qwen/Qwen2.5-1.5B-Instruct


In [5]:
# Core utilities

LETTERS="ABCDEFGHIJKLMNOPQRSTUVWXYZ"
NEXUS_FRAME = "\n".join([
    "Use the Nexus operational lens.",
    "",
    "Rules:",
    "1. Prefer verbs/operations over nouns/labels.",
    "2. Treat shape, constraint, boundary, and gap as primary.",
    "3. A good answer preserves function while hiding complexity inward.",
    "4. For repair questions, start from the needed future state and work backward.",
    "5. Do not choose surface similarity when operational fit is missing.",
    "6. Choose the single best collapse.",
])

def maybe_chat(prompt):
    if USE_CHAT_TEMPLATE and hasattr(tokenizer,"apply_chat_template"):
        try:
            return tokenizer.apply_chat_template([{"role":"user","content":prompt}], tokenize=False, add_generation_prompt=True)
        except Exception:
            pass
    return prompt

def generate_text(prompt, max_new_tokens=512, temperature=0.0):
    rendered=maybe_chat(prompt)
    inputs=tokenizer(rendered, return_tensors="pt").to(DEVICE)
    gen_kwargs=dict(max_new_tokens=max_new_tokens, do_sample=temperature>0, pad_token_id=tokenizer.eos_token_id)
    if temperature>0:
        gen_kwargs["temperature"]=temperature
        gen_kwargs["top_p"]=0.9
    with torch.no_grad():
        out=lm.generate(**inputs, **gen_kwargs)
    new_tokens=out[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def conditional_logprob(prefix,suffix):
    prefix_ids=tokenizer(prefix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    full_ids=tokenizer(prefix+suffix, return_tensors="pt", add_special_tokens=False)["input_ids"].to(DEVICE)
    with torch.no_grad():
        out=lm(full_ids)
        logits=out.logits[:,:-1,:]
        targets=full_ids[:,1:]
        lp=F.log_softmax(logits,dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    start=max(prefix_ids.shape[1]-1,0)
    return float(lp[:,start:].mean().item())

def zscore_local(x):
    x=np.array(x,dtype=np.float32)
    return (x-x.mean())/(x.std()+1e-8)

def margin_of(scores):
    order=np.sort(np.array(scores))[::-1]
    return float(order[0]-order[1]) if len(order)>1 else 0.0

def argmax_margin(scores):
    return int(np.argmax(scores)), margin_of(scores)

def build_mcq_prompt(row):
    parts=[NEXUS_FRAME,"","Task:",row["prompt"].strip(),"","Choices:"]
    for i,c in enumerate(row["choices"]):
        parts.append(f"{LETTERS[i]}. {c}")
    parts += ["","Return only the single best letter."]
    return "\n".join(parts)

def score_base(row):
    p=build_mcq_prompt(row)
    rendered=maybe_chat(p)
    scores=np.array([conditional_logprob(rendered," "+LETTERS[i]) for i in range(len(row["choices"]))], dtype=np.float32)
    pred=int(np.argmax(scores))
    prob=np.exp(scores-scores.max()); prob=prob/(prob.sum()+1e-8)
    ent=float(-np.sum(prob*np.log(prob+1e-8)))
    return p,scores,prob.astype(np.float32),pred,margin_of(scores),ent

def score_answer_text_with_choices(row,prompt_text):
    prefix=maybe_chat(prompt_text+"\n\nThe Nexus collapse is")
    return np.array([conditional_logprob(prefix," "+choice) for choice in row["choices"]], dtype=np.float32)

def encode_norm(texts):
    return embedder.encode(texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype(np.float32)

def cos_scores(choices, target_text):
    cand_vecs=encode_norm(choices)
    target_vec=encode_norm([target_text])[0]
    return (cand_vecs @ target_vec).astype(np.float32)

STOP=set("a an the and or but if then than to of in on for with without into from by as is are was were be being been it this that these those only not no yes because while under over through across below above what which who when where why how does do did".split())

def toks(s):
    return [t for t in re.findall(r"[a-z0-9]+", str(s).lower()) if t not in STOP and len(t)>1]

def jaccard(a,b):
    A=set(toks(a)); B=set(toks(b))
    if not A or not B:
        return 0.0
    return len(A&B)/len(A|B)


In [6]:
# Compiled-slot baseline helpers

@dataclass
class NeedSlot:
    required_operation: str
    preserved_function: str
    boundary_conditions: List[str]
    anti_fits: List[str]
    admissible_shape: str
    failure_modes: List[str]
    source: str = "abstract_compiler"

    def positive_text(self) -> str:
        return "\n".join([self.required_operation,self.preserved_function,self.admissible_shape,*self.boundary_conditions])
    def negative_text(self) -> str:
        return "\n".join([*self.anti_fits,*self.failure_modes])
    def no_admissible_text(self) -> str:
        return "\n".join([self.required_operation,self.preserved_function,*self.boundary_conditions])
    def admissible_text(self) -> str:
        return "\n".join([self.admissible_shape,self.required_operation,self.preserved_function])

def slot_from_raw(raw):
    return NeedSlot(
        required_operation=raw["required_operation"],
        preserved_function=raw["preserved_function"],
        boundary_conditions=list(raw["boundary_conditions"]),
        anti_fits=list(raw["anti_fits"]),
        admissible_shape=raw["admissible_shape"],
        failure_modes=list(raw["failure_modes"]),
    )

slot_by_base_id={row["id"]: slot_from_raw(ABSTRACT_SLOT_SPECS_RAW[row["id"]]) for row in base_rows}
slots_df=pd.DataFrame([{"base_id":k, **asdict(v)} for k,v in slot_by_base_id.items()])
display(slots_df[["base_id","required_operation","admissible_shape"]].head(24))

def score_compiled_slot_branches(row, slot: NeedSlot):
    choices=row["choices"]
    pos_cos=cos_scores(choices, slot.positive_text())
    neg_cos=cos_scores(choices, slot.negative_text())
    noadm_cos=cos_scores(choices, slot.no_admissible_text())
    adm_cos=cos_scores(choices, slot.admissible_text())

    pos_j=np.array([jaccard(c, slot.positive_text()) for c in choices], dtype=np.float32)
    neg_j=np.array([jaccard(c, slot.negative_text()) for c in choices], dtype=np.float32)
    noadm_j=np.array([jaccard(c, slot.no_admissible_text()) for c in choices], dtype=np.float32)
    adm_j=np.array([jaccard(c, slot.admissible_text()) for c in choices], dtype=np.float32)

    positive=(0.60*zscore_local(pos_cos)+0.40*zscore_local(pos_j)).astype(np.float32)
    admissible=(0.60*zscore_local(adm_cos)+0.40*zscore_local(adm_j)).astype(np.float32)
    anti_avoid=(-0.60*zscore_local(neg_cos)-0.40*zscore_local(neg_j)).astype(np.float32)
    no_admissible=(0.70*zscore_local(noadm_cos)+0.30*zscore_local(noadm_j)).astype(np.float32)
    full=(0.25*admissible + 0.35*positive + 0.40*anti_avoid).astype(np.float32)
    return {"positive":positive,"admissible":admissible,"anti_avoid":anti_avoid,"no_admissible":no_admissible,"full":full}

def support_for_pred_from_branches(branches, pred):
    return (
        int(np.argmax(branches["positive"])==pred)
        + int(np.argmax(branches["anti_avoid"])==pred)
        + int(np.argmax(branches["no_admissible"])==pred)
        + int(np.argmax(branches["full"])==pred)
    )


,base_id,required_operation,admissible_shape
0,adv_coupler_01,restore centered torque transfer by adding ela...,elastic concentric compression around a round ...
1,adv_coupler_02,restore centered compressive coupling with min...,removable symmetric compression element
2,adv_car_01,collapse hidden vehicle machinery into usable ...,human-facing control interface over hidden veh...
3,adv_house_01,fold environmental loads utilities privacy and...,habitation interface over weather load heat ut...
4,adv_api_01,hide service machinery below one stable callab...,stable interface over hidden implementation co...
5,adv_llm_01,describe sequential token growth through an ev...,stateful incremental generator that emits and ...
6,adv_sha_01,treat digest as residue of deterministic folding,deterministic compressed folding residue
7,adv_sha_02,identify a shared role as stored bias during f...,stored structural control bias inside a foldin...
8,adv_observable_01,provide readable residues inside recursion,readable recursive residue or observable trace
9,adv_breath_01,change state through pressure tolerance or adm...,nonmaterial adjustment of admissible transitio...


In [7]:
# Initial self-generating slot constructor

REQUIRED_CONTRACT_KEYS=[
    "family_class","domain_carrier","forbidden_neighbor_carrier","boundary_conditions",
    "preserved_function","failure_modes","witness_readout","residue"
]

def contract_prompt(row):
    return f"""
You are the Nexus Slot Constructor.

Build the missing-shape contract BEFORE answer selection.
You must output strict JSON only. No markdown. No explanation.
Do NOT answer the task. Do NOT mention answer choices.

Prompt:
{row["prompt"]}

Return one JSON object with these exact keys:
family_class
domain_carrier
forbidden_neighbor_carrier
boundary_conditions
preserved_function
failure_modes
witness_readout
residue

Rules:
- family_class must be a concrete operational family, never a placeholder.
- domain_carrier must contain prompt-specific operational markers.
- forbidden_neighbor_carrier must contain nearby wrong collision families.
- boundary_conditions must define what a valid collapse must preserve and reject.
- preserved_function must state the function that remains active.
- failure_modes must name concrete wrong collapses.
- witness_readout must be observable from the prompt's operational field.
- Forbidden phrases: "general operation family", "operation family", "data processing", "as a concise phrase".
- JSON only.
""".strip()

def extract_json_object(text):
    text=text.strip()
    text=re.sub(r"^```(?:json)?", "", text.strip(), flags=re.I).strip()
    text=re.sub(r"```$", "", text.strip()).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start=text.find("{"); end=text.rfind("}")
    if start>=0 and end>start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    return None

def normalize_contract(obj, row_id="unknown"):
    if not isinstance(obj, dict):
        obj={}
    out={k:obj.get(k,None) for k in REQUIRED_CONTRACT_KEYS}

    def listify(x):
        if x is None:
            return []
        if isinstance(x, list):
            return [str(v).strip() for v in x if str(v).strip()]
        if isinstance(x, str):
            if ";" in x:
                return [p.strip() for p in x.split(";") if p.strip()]
            return [x.strip()] if x.strip() else []
        return [str(x).strip()]

    out["family_class"]=str(out["family_class"] or "").strip()
    out["domain_carrier"]=listify(out["domain_carrier"])
    out["forbidden_neighbor_carrier"]=listify(out["forbidden_neighbor_carrier"])
    out["boundary_conditions"]=listify(out["boundary_conditions"])
    out["preserved_function"]=str(out["preserved_function"] or "").strip()
    out["failure_modes"]=listify(out["failure_modes"])
    out["witness_readout"]=str(out["witness_readout"] or "").strip()
    out["residue"]=out["residue"] if out["residue"] not in ["","null","None"] else None
    out["source"]="generated_slot_constructor"
    out["row_id"]=row_id

    # Minimal fallback; this is not a domain schema.
    if not out["family_class"]:
        out["family_class"]="missing operational shape"
    if not out["domain_carrier"]:
        out["domain_carrier"]=toks(base_by_id.get(row_id,{}).get("prompt",""))[:8]
    if not out["boundary_conditions"]:
        out["boundary_conditions"]=["preserve the function requested by the prompt"]
    if not out["preserved_function"]:
        out["preserved_function"]="preserve the prompt required operation"
    if not out["failure_modes"]:
        out["failure_modes"]=["surface label match without operational fit"]
    if not out["witness_readout"]:
        out["witness_readout"]="candidate satisfies the prompt-specific function under constraint"
    return out

def contract_to_text(contract, mode="full"):
    if mode=="positive":
        parts=[contract.get("family_class",""), contract.get("preserved_function",""),
               " ".join(contract.get("domain_carrier",[])), " ".join(contract.get("boundary_conditions",[])),
               contract.get("witness_readout","")]
    elif mode=="negative":
        parts=[" ".join(contract.get("forbidden_neighbor_carrier",[])), " ".join(contract.get("failure_modes",[]))]
    elif mode=="domain":
        parts=contract.get("domain_carrier",[])
    elif mode=="forbidden":
        parts=contract.get("forbidden_neighbor_carrier",[])
    elif mode=="witness":
        parts=[contract.get("witness_readout","")]
    else:
        parts=[contract.get("family_class",""), contract.get("preserved_function",""),
               " ".join(contract.get("domain_carrier",[])), " ".join(contract.get("forbidden_neighbor_carrier",[])),
               " ".join(contract.get("boundary_conditions",[])), " ".join(contract.get("failure_modes",[])),
               contract.get("witness_readout","")]
    return "\n".join([str(p) for p in parts if str(p).strip()])

def generate_contract_for_row(row, force=False):
    cache_dir=Path(OUTPUT_DIR)/"contracts"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path=cache_dir/f"{row['id']}.json"
    raw_path=cache_dir/f"{row['id']}.raw.txt"

    if cache_path.exists() and not force:
        try:
            return normalize_contract(json.loads(cache_path.read_text(encoding="utf-8")), row_id=row["id"])
        except Exception:
            pass

    prompt=contract_prompt(row)
    obj=None; raw_text=""
    for attempt in range(CONTRACT_REPAIR_MAX_ATTEMPTS+1):
        raw_text=generate_text(
            prompt if attempt==0 else prompt + "\n\nPrevious output was invalid. Return strict JSON only.",
            max_new_tokens=CONTRACT_MAX_NEW_TOKENS,
            temperature=CONTRACT_TEMPERATURE,
        )
        obj=extract_json_object(raw_text)
        if obj is not None:
            break

    contract=normalize_contract(obj, row_id=row["id"])
    cache_path.write_text(json.dumps(contract, indent=2), encoding="utf-8")
    raw_path.write_text(raw_text, encoding="utf-8")
    return contract

contracts={}
for row in tqdm(base_rows, desc="Generating/loading v68 initial slot contracts"):
    contracts[row["id"]]=generate_contract_for_row(row, force=REGENERATE_CONTRACTS)

contracts_df=pd.DataFrame([
    {
        "base_id":sid,
        "family_class":c["family_class"],
        "domain_carrier":" | ".join(c["domain_carrier"]),
        "forbidden_neighbor_carrier":" | ".join(c["forbidden_neighbor_carrier"]),
        "preserved_function":c["preserved_function"],
        "witness_readout":c["witness_readout"],
        "residue":c["residue"],
    }
    for sid,c in contracts.items()
])
display(contracts_df)
contracts_df.to_csv(Path(OUTPUT_DIR)/"initial_contracts.csv", index=False)


Generating/loading v68 initial slot contracts:   0%|          | 0/24 [00:00<?, ?it/s]

,base_id,family_class,domain_carrier,forbidden_neighbor_carrier,preserved_function,witness_readout,residue
0,adv_coupler_01,spinning,"rubber coupler, vacuum pump shaft",concrete | metal,rotation transmission,observed radial compression without coupler mo...,tightened rubber coupler
1,adv_coupler_02,compressive coupling,motion,flexible joint,compressive coupling,the system exhibits consistent behavior during...,no change in overall structure or functionality
2,adv_car_01,interface-collapse,car,combustion,combustion,"a car hiding combustion, gearing, sensors, tir...",the remaining components of the car after the ...
3,adv_house_01,structural,inward fold,door,remaining structural integrity,internal crease visible upon inspection,collapsed inward fold material
4,adv_api_01,missing-shape,API call,authentication | routing | validation | persis...,operational event,an API call exposing one method while hiding a...,NaN
5,adv_llm_01,Learner,LLM_answer,general operation family,interpretation of input tokens,the sequence of tokens produced,remaining tokens after exhaustion or error
6,adv_sha_01,concrete,digest SHA-256,general operation family,hashing algorithm integrity preservation,observable through cryptographic hash functions,the resulting binary string after hashing
7,adv_sha_02,rhyme,roles,identical,rhyming roles,identical roles,identical
8,adv_observable_01,recursive,loop,hidden state,load,observable from the prompt's operational field,valid collapse
9,adv_breath_01,recursive system,breathes,moving matter,functionality of breathing,no visible change in shape or motion,unchanged structure


In [8]:
# v68 five-dimensional operational critic + recursive repair

OP_DIMS=["need","function","boundary","trap","collapse"]
OP_DIM_DESCRIPTIONS={
    "need":"occupies the inverse cavity created by the prompt",
    "function":"preserves or redirects the required operation",
    "boundary":"respects constraint boundaries such as motion, heat, pressure, permissions, timing, or context",
    "trap":"rejects noun labels and surface similarity traps",
    "collapse":"produces one executable interface rather than many vague interpretations",
}

PLACEHOLDER_PATTERNS=[
    "general operation family",
    "operation family",
    "data processing and manipulation",
    "data processing and transformation",
    "mathematical operation",
    "operation",
    "general operation",
    "as a concise phrase",
]
GENERIC_CONTRACT_TERMS=set("""
operation family data processing manipulation transformation object system thing concept answer choice
prompt response function valid invalid correct incorrect shape nexus recursive fold folding general concise phrase
mathematics geometry topology algorithm programming software hardware component process structure analysis
label surface category generic formal symbolic description
""".split())

DOMAIN_MIN_TERMS=4
FORBIDDEN_MIN_TERMS=3
WITNESS_MIN_TERMS=5
BOUNDARY_MIN_TERMS=3
FAILURE_MIN_TERMS=2

def text_terms(x):
    if isinstance(x, list):
        x=" ".join(map(str,x))
    return [t for t in toks(str(x)) if len(t)>2]

def prompt_keyword_carrier_from_text(prompt, limit=12):
    raw=toks(prompt)
    bad=set(GENERIC_CONTRACT_TERMS) | {"candidate","answer","which","what","does","must"}
    kept=[]
    for t in raw:
        if t not in bad and t not in kept:
            kept.append(t)
    return kept[:limit]

def contract_prompt_terms(row):
    return set(text_terms(row["prompt"]))

def lexical_contract_stats(row, contract):
    prompt_terms=contract_prompt_terms(row)
    family=str(contract.get("family_class","")).lower().strip()
    domain_terms=text_terms(contract.get("domain_carrier",[]))
    forbidden_terms=text_terms(contract.get("forbidden_neighbor_carrier",[]))
    boundary_terms=text_terms(contract.get("boundary_conditions",[]))
    preserved_terms=text_terms(contract.get("preserved_function",""))
    failure_terms=text_terms(contract.get("failure_modes",[]))
    witness_terms=text_terms(contract.get("witness_readout",""))

    domain_set=set(domain_terms)
    forbidden_set=set(forbidden_terms)
    witness_set=set(witness_terms)
    all_contract_terms=set(domain_terms+forbidden_terms+boundary_terms+preserved_terms+failure_terms+witness_terms+text_terms(family))

    placeholder=any(p in family for p in PLACEHOLDER_PATTERNS)
    generic_domain_frac=sum(t in GENERIC_CONTRACT_TERMS for t in domain_terms)/max(len(domain_terms),1)
    generic_forbidden_frac=sum(t in GENERIC_CONTRACT_TERMS for t in forbidden_terms)/max(len(forbidden_terms),1)

    return {
        "placeholder":placeholder,
        "domain_len":len(domain_set),
        "forbidden_len":len(forbidden_set),
        "witness_len":len(witness_set),
        "boundary_len":len(set(boundary_terms)),
        "failure_len":len(set(failure_terms)),
        "domain_prompt_overlap":len(domain_set & prompt_terms)/max(len(domain_set),1),
        "witness_prompt_overlap":len(witness_set & prompt_terms)/max(len(witness_set),1),
        "contract_prompt_overlap":len(all_contract_terms & prompt_terms)/max(len(all_contract_terms),1),
        "domain_forbidden_overlap":len(domain_set & forbidden_set)/max(len(domain_set | forbidden_set),1),
        "generic_domain_frac":generic_domain_frac,
        "generic_forbidden_frac":generic_forbidden_frac,
    }

OP_POS_ANCHORS={
    "need":"occupies the inverse cavity; fills the missing operational shape; satisfies the exact need created by the failure",
    "function":"preserves the required verb; keeps the operation running; redirects function without breaking it",
    "boundary":"respects constraint boundaries; holds under the relevant motion, pressure, timing, permissions, and context",
    "trap":"rejects noun traps; avoids surface labels; refuses semantic resemblance without operational fit",
    "collapse":"collapses many constraints into one executable interface; selects one coherent functional closure",
}
OP_NEG_ANCHORS={
    "need":"fills space without satisfying the missing shape; names something nearby but misses the inverse cavity",
    "function":"stops the verb; names the function but does not preserve the operation",
    "boundary":"violates constraints; overbinds or underbinds; ignores heat, motion, pressure, permission, timing, or context",
    "trap":"falls for surface labels; chooses a technical noun; matches words without fit",
    "collapse":"leaves multiple interpretations; vague description; no single executable interface",
}

anchor_vecs={}
for dim in OP_DIMS:
    anchor_vecs[(dim,"pos")]=encode_norm([OP_POS_ANCHORS[dim]])[0]
    anchor_vecs[(dim,"neg")]=encode_norm([OP_NEG_ANCHORS[dim]])[0]

def dim_text(row, contract, dim):
    prompt=row["prompt"]
    if dim=="need":
        return "\n".join([
            "Prompt:", prompt,
            "Family:", contract.get("family_class",""),
            "Domain carrier:", " ".join(contract.get("domain_carrier",[])),
            "Witness:", contract.get("witness_readout",""),
            "Question: does this occupy the inverse cavity created by the prompt?"
        ])
    if dim=="function":
        return "\n".join([
            "Prompt:", prompt,
            "Preserved function:", contract.get("preserved_function",""),
            "Boundary conditions:", " ".join(contract.get("boundary_conditions",[])),
            "Question: does this preserve or redirect the required operation?"
        ])
    if dim=="boundary":
        return "\n".join([
            "Prompt:", prompt,
            "Boundary conditions:", " ".join(contract.get("boundary_conditions",[])),
            "Failure modes:", " ".join(contract.get("failure_modes",[])),
            "Question: does this respect the constraint boundary?"
        ])
    if dim=="trap":
        return "\n".join([
            "Prompt:", prompt,
            "Forbidden neighbor carrier:", " ".join(contract.get("forbidden_neighbor_carrier",[])),
            "Failure modes:", " ".join(contract.get("failure_modes",[])),
            "Question: does this reject surface labels and noun traps?"
        ])
    if dim=="collapse":
        return "\n".join([
            "Prompt:", prompt,
            "Family:", contract.get("family_class",""),
            "Witness:", contract.get("witness_readout",""),
            "Boundary conditions:", " ".join(contract.get("boundary_conditions",[])),
            "Question: does this produce one executable interface?"
        ])
    raise ValueError(dim)

def operational_dim_score(row, contract, dim):
    txt=dim_text(row,contract,dim)
    vec=encode_norm([txt])[0]
    pos=float(vec @ anchor_vecs[(dim,"pos")])
    neg=float(vec @ anchor_vecs[(dim,"neg")])
    raw=pos-neg
    stats=lexical_contract_stats(row,contract)

    if dim=="need":
        support=0.35*min(stats["domain_len"]/8,1.0)+0.35*min(stats["domain_prompt_overlap"]/0.35,1.0)+0.30*min(stats["witness_len"]/10,1.0)
    elif dim=="function":
        support=0.35*min(len(set(text_terms(contract.get("preserved_function",""))))/8,1.0)+0.35*min(stats["boundary_len"]/6,1.0)+0.30*min(stats["contract_prompt_overlap"]/0.35,1.0)
    elif dim=="boundary":
        support=0.45*min(stats["boundary_len"]/6,1.0)+0.30*min(stats["failure_len"]/4,1.0)+0.25*(1.0-min(stats["domain_forbidden_overlap"]/0.5,1.0))
    elif dim=="trap":
        support=0.45*min(stats["forbidden_len"]/6,1.0)+0.30*min(stats["failure_len"]/4,1.0)+0.25*(1.0-min(stats["generic_domain_frac"],1.0))
    elif dim=="collapse":
        support=0.40*min(stats["witness_len"]/10,1.0)+0.30*min(stats["boundary_len"]/6,1.0)+0.30*(0.0 if stats["placeholder"] else 1.0)
    emb_score=1.0/(1.0+math.exp(-8.0*raw))
    score=float(0.55*emb_score + 0.45*support)
    return max(0.0,min(1.0,score)), {"pos":pos, "neg":neg, "raw":raw, "support":support, "emb_score":emb_score}

def operational_audit(row, contract):
    stats=lexical_contract_stats(row,contract)
    dims={}
    diag={}
    for dim in OP_DIMS:
        s,d=operational_dim_score(row,contract,dim)
        dims[dim]=s
        diag[dim]=d

    failures=[]
    if dims["need"] < 0.55: failures.append("need_failure")
    if dims["function"] < 0.55: failures.append("function_failure")
    if dims["boundary"] < 0.55: failures.append("boundary_failure")
    if dims["trap"] < 0.55: failures.append("trap_failure")
    if dims["collapse"] < 0.55: failures.append("collapse_failure")

    if stats["placeholder"]: failures.append("collapse_failure:placeholder_family_class")
    if stats["domain_len"] < DOMAIN_MIN_TERMS or stats["domain_prompt_overlap"] < 0.15:
        failures.append("need_failure:domain_not_prompt_anchored")
    if stats["forbidden_len"] < FORBIDDEN_MIN_TERMS:
        failures.append("trap_failure:forbidden_too_thin")
    if stats["boundary_len"] < BOUNDARY_MIN_TERMS:
        failures.append("boundary_failure:boundary_too_thin")
    if stats["failure_len"] < FAILURE_MIN_TERMS:
        failures.append("trap_failure:failure_modes_too_thin")
    if stats["witness_len"] < WITNESS_MIN_TERMS:
        failures.append("collapse_failure:witness_too_thin")
    if stats["domain_forbidden_overlap"] > 0.30:
        failures.append("trap_failure:domain_forbidden_overlap")

    seen=set()
    failures=[x for x in failures if not (x in seen or seen.add(x))]

    vals=np.array([dims[d] for d in OP_DIMS], dtype=np.float32)
    harmonic=float(len(vals)/np.sum(1.0/(vals+1e-6)))
    mean=float(vals.mean())
    minv=float(vals.min())
    quality=float(0.50*harmonic + 0.30*mean + 0.20*minv)
    quality=max(0.0, min(1.0, quality - 0.025*len(failures)))
    return {"quality":quality, "failures":failures, "dims":dims, "diag":diag, **stats}

def audit_df_for(contracts_map, label):
    rows_a=[]
    for row in base_rows:
        c=contracts_map[row["id"]]
        a=operational_audit(row,c)
        rows_a.append({
            "base_id":row["id"],
            "label":label,
            "quality":a["quality"],
            "F_need":a["dims"]["need"],
            "F_function":a["dims"]["function"],
            "F_boundary":a["dims"]["boundary"],
            "F_trap":a["dims"]["trap"],
            "F_collapse":a["dims"]["collapse"],
            "failures":" | ".join(a["failures"]),
            "family_class":c.get("family_class",""),
            "domain_carrier":" | ".join(c.get("domain_carrier",[])),
            "forbidden_neighbor_carrier":" | ".join(c.get("forbidden_neighbor_carrier",[])),
            "boundary_conditions":" | ".join(c.get("boundary_conditions",[])),
            "preserved_function":c.get("preserved_function",""),
            "failure_modes":" | ".join(c.get("failure_modes",[])),
            "witness_readout":c.get("witness_readout",""),
        })
    return pd.DataFrame(rows_a)

initial_audit_df=audit_df_for(contracts,"initial")
display(initial_audit_df.sort_values("quality").head(24))
print("initial mean operational quality:", initial_audit_df["quality"].mean())
initial_audit_df.to_csv(Path(OUTPUT_DIR)/"initial_operational_audit.csv", index=False)


,base_id,label,quality,F_need,F_function,F_boundary,F_trap,F_collapse,failures,family_class,domain_carrier,forbidden_neighbor_carrier,boundary_conditions,preserved_function,failure_modes,witness_readout
7,adv_sha_02,initial,0.390080,0.554145,0.479711,0.627981,0.652282,0.489007,function_failure | collapse_failure | need_fai...,rhyme,roles,identical,identical roles,rhyming roles,non-rhyming roles,identical roles
22,adv_weight_01,initial,0.407227,0.555991,0.539072,0.649609,0.618801,0.441781,function_failure | collapse_failure | need_fai...,lookup_table,LLM_weight_field,database,valid_LLM_weights,weight_update,invalid_weight_values,LLM_performance
12,adv_tree_01,initial,0.442559,0.527686,0.689628,0.830546,0.696544,0.408973,need_failure | collapse_failure | need_failure...,tree,exposes,bush | vine,"{'preserve': ['leaf', 'trunk', 'fruit'], 'reje...",photosynthesis,rotting leaves | falling fruits,shade
11,adv_flower_01,initial,0.449391,0.531788,0.719776,0.725601,0.603472,0.472118,need_failure | collapse_failure | need_failure...,visible bloom,flower,surface color,"{'preserve': ['visible bloom'], 'reject': []}",remaining as a visible bloom,disappearing or becoming invisible,the presence of petals and stamens
16,adv_ping_01,initial,0.476927,0.582869,0.610420,0.836351,0.585497,0.469221,collapse_failure | need_failure:domain_not_pro...,math,ping,beacon,"{'preserve': 'shape', 'reject': 'non-beacon'}",being tested,colliding with non-ping shapes | not conformin...,the presence of a distinct pattern or structure
14,adv_surface_02,initial,0.476994,0.439344,0.537458,0.870276,0.665048,0.644524,need_failure | function_failure | need_failure...,Fold,a shape in space-time,Translation | Rotation,The shape must maintain its volume under folding.,Preserves the original shape's volume.,Shape becomes distorted beyond recognition | V...,Witnesses can observe the change in volume as ...
9,adv_breath_01,initial,0.488137,0.606088,0.641249,0.789855,0.648763,0.464961,collapse_failure | need_failure:domain_not_pro...,recursive system,breathes,moving matter,"{'preserve': 'without', 'reject': 'moving'}",functionality of breathing,collapse due to movement | failure in maintain...,no visible change in shape or motion
10,adv_fold_01,initial,0.488376,0.684000,0.578716,0.633299,0.506016,0.665572,trap_failure | need_failure:domain_not_prompt_...,folding,chair,non-folding,{'seated_function': 'return'},fold,unfolded,the chair returns to its original shape after ...
19,adv_solution_01,initial,0.489589,0.650233,0.623649,0.784701,0.574786,0.488007,collapse_failure | need_failure:domain_not_pro...,solution,under the Nexus lens,answer | string,"{'preserve': ['functionality'], 'reject': []}",remaining functionality,surface label match without operational fit,observable from the prompt's operational field
20,adv_idea_01,initial,0.518453,0.593737,0.677934,0.715653,0.672385,0.544648,collapse_failure | need_failure:domain_not_pro...,handle,carryable,contradiction,"{'preserve': ['hidden contradictions'], 'rejec...",usefulness,analogies | compression,a carryable handle


initial mean operational quality: 0.5438852230707805


In [9]:
# v68 recursive operational repair

def failure_axes_from_operational_audit(audit):
    failures=audit.get("failures",[])
    axes=[]
    if any(f.startswith("need_failure") for f in failures):
        axes += ["need","domain_carrier"]
    if any(f.startswith("function_failure") for f in failures):
        axes += ["function","preserved_function"]
    if any(f.startswith("boundary_failure") for f in failures):
        axes += ["boundary","boundary_conditions"]
    if any(f.startswith("trap_failure") for f in failures):
        axes += ["trap","forbidden_neighbor_carrier","failure_modes"]
    if any(f.startswith("collapse_failure") for f in failures):
        axes += ["collapse","witness_readout","family_class"]
    seen=set()
    axes=[x for x in axes if not (x in seen or seen.add(x))]
    if not axes and audit.get("quality",0.0)<OP_LOCK:
        axes.append("operational_densification")
    return axes

def operational_signature(audit):
    return {
        "quality":round(float(audit["quality"]),4),
        "F_need":round(float(audit["dims"]["need"]),4),
        "F_function":round(float(audit["dims"]["function"]),4),
        "F_boundary":round(float(audit["dims"]["boundary"]),4),
        "F_trap":round(float(audit["dims"]["trap"]),4),
        "F_collapse":round(float(audit["dims"]["collapse"]),4),
        "failures":list(audit.get("failures",[])),
    }

def compact_contract_view(contract):
    return {k:contract.get(k,None) for k in REQUIRED_CONTRACT_KEYS}

def trace_summary(trace, max_items=4):
    if not trace:
        return "[]"
    slim=[]
    for t in trace[-max_items:]:
        slim.append({
            "depth":t["depth"],
            "quality":round(float(t.get("quality",0.0)),4),
            "F_need":round(float(t.get("F_need",0.0)),4),
            "F_function":round(float(t.get("F_function",0.0)),4),
            "F_boundary":round(float(t.get("F_boundary",0.0)),4),
            "F_trap":round(float(t.get("F_trap",0.0)),4),
            "F_collapse":round(float(t.get("F_collapse",0.0)),4),
            "accepted":bool(t.get("accepted",False)),
            "failures":t.get("failures",[]),
            "collapse":t.get("collapse_reason",""),
        })
    return json.dumps(slim, indent=2)

def recursive_operational_repair_prompt(row, contract, audit, trace, depth):
    axes=failure_axes_from_operational_audit(audit)
    failure_text="; ".join(audit["failures"]) if audit["failures"] else "none"
    signature=json.dumps(operational_signature(audit), indent=2)
    history=trace_summary(trace)

    return f"""
You are the Nexus Recursive Operational Slot Constructor.

Do NOT answer the task.
Do NOT mention answer choices.
Repair the missing-shape contract only.

Recurrence:
C_{depth} -> A_{depth} -> Ω_{depth} -> ΔC_{depth} -> C_{depth+1}

Prompt:
{row["prompt"]}

Current contract C_{depth}:
{json.dumps(compact_contract_view(contract), indent=2)}

Operational audit A_{depth}:
{signature}

Ω operational failures:
{failure_text}

Failed operational axes to repair:
{", ".join(axes)}

Recent recursion trace:
{history}

Five operational dimensions:
1. NEED: occupy the inverse cavity created by the prompt.
2. FUNCTION: preserve or redirect the required operation.
3. BOUNDARY: respect constraint boundaries.
4. TRAP: reject noun/surface-label confusion.
5. COLLAPSE: produce one executable interface.

Rules:
- Candidate-blind: do not ask for or infer answer choices.
- Preserve good structure from C_{depth}.
- Repair only failed operational axes.
- Use prompt-specific carriers.
- Avoid generic phrases like "general operation family" or "data processing".
- Strict JSON only.

Return exactly this JSON object:
{{
  "family_class": "...",
  "domain_carrier": ["...", "...", "...", "...", "..."],
  "forbidden_neighbor_carrier": ["...", "...", "...", "..."],
  "boundary_conditions": ["...", "...", "...", "..."],
  "preserved_function": "...",
  "failure_modes": ["...", "...", "..."],
  "witness_readout": "...",
  "residue": null
}}
""".strip()

def deterministic_operational_scaffold(row, contract, audit):
    c=json.loads(json.dumps(compact_contract_view(contract)))
    axes=failure_axes_from_operational_audit(audit)
    prompt_terms=prompt_keyword_carrier_from_text(row["prompt"], limit=18)
    prompt_phrase=" ".join(prompt_terms[:8]) if prompt_terms else "the prompt field"

    if "collapse" in axes or "family_class" in axes or any(p in str(c.get("family_class","")).lower() for p in PLACEHOLDER_PATTERNS):
        c["family_class"]=f"operational closure of {prompt_phrase}"

    if "need" in axes or "domain_carrier" in axes:
        merged=[]
        for item in c.get("domain_carrier",[]):
            for t in text_terms(item):
                if t not in GENERIC_CONTRACT_TERMS and t not in merged:
                    merged.append(t)
        for t in prompt_terms:
            if t not in merged:
                merged.append(t)
        c["domain_carrier"]=merged[:max(DOMAIN_MIN_TERMS+5,9)]

    if "function" in axes or "preserved_function" in axes or not str(c.get("preserved_function","")).strip():
        c["preserved_function"]="preserve the operation implied by " + ", ".join(prompt_terms[:6]) if prompt_terms else "preserve the prompt-specific operation"

    if "boundary" in axes or "boundary_conditions" in axes:
        boundaries=list(c.get("boundary_conditions",[]))
        seed=[
            "must preserve the prompt-specific operation",
            "must not collapse into surface label similarity",
            "must obey the prompt's physical, logical, social, or contextual constraints",
            "must reject nearby carriers that do not satisfy the required function",
        ]
        joined=" ".join(prompt_terms)
        if any(t in joined for t in ["coupler","shaft","compression","motion","pump","repair","rubber"]):
            seed += ["must hold centered radial compression under rotation", "must not overbind or stop motion"]
        if any(t in joined for t in ["tree","leaf","trunk","fruit","shade"]):
            seed += ["must remain biological and plant-operational", "must not collapse into silicon or shelter interface logic"]
        for x in seed:
            if x not in boundaries:
                boundaries.append(x)
        c["boundary_conditions"]=boundaries[:10]

    if "trap" in axes or "forbidden_neighbor_carrier" in axes:
        forbidden=list(c.get("forbidden_neighbor_carrier",[]))
        seed=[
            "surface label without operational fit",
            "generic wording without preserved function",
            "wrong neighboring domain carrier",
            "answer-choice resemblance without boundary satisfaction",
        ]
        joined=" ".join(prompt_terms)
        if any(t in joined for t in ["coupler","shaft","compression","motion","pump","repair","rubber"]):
            seed += ["string wraps but does not center under rotation", "loose under-binding", "permanent over-binding", "name-only rubber-part match"]
        if any(t in joined for t in ["tree","leaf","trunk","fruit","shade"]):
            seed += ["chip miniaturization carrier", "Moore switching density carrier", "house shelter carrier", "generic hidden-interface carrier"]
        if any(t in joined for t in ["llm","token","indexed","state"]):
            seed += ["stored final paragraph", "static database lookup", "random text without state update"]
        if any(t in joined for t in ["sha","digest","constants","weights"]):
            seed += ["semantic meaning extraction", "database pointer", "constant value mysticism"]
        for x in seed:
            if x not in forbidden:
                forbidden.append(x)
        c["forbidden_neighbor_carrier"]=forbidden[:10]

    if "trap" in axes or "failure_modes" in axes:
        failures=list(c.get("failure_modes",[]))
        for x in ["surface wording wins over operational fit", "nearby family collision replaces the real domain carrier", "function is named but not preserved"]:
            if x not in failures:
                failures.append(x)
        c["failure_modes"]=failures[:8]

    if "collapse" in axes or "witness_readout" in axes:
        c["witness_readout"]=("the chosen collapse preserves " + ", ".join(prompt_terms[:6]) + " while rejecting surface-label and wrong-carrier matches") if prompt_terms else "the chosen collapse preserves the prompt-specific function"

    c["residue"]=None
    return normalize_contract(c,row_id=row["id"])

def accept_operational_candidate(current_audit, candidate_audit):
    cur_fail=set(current_audit.get("failures",[]))
    new_fail=set(candidate_audit.get("failures",[]))
    removed=len(cur_fail-new_fail)
    added=len(new_fail-cur_fail)
    dq=float(candidate_audit["quality"]-current_audit["quality"])
    dim_improve=sum(candidate_audit["dims"][d] > current_audit["dims"][d] + 0.025 for d in OP_DIMS)
    dim_decline=sum(candidate_audit["dims"][d] < current_audit["dims"][d] - 0.050 for d in OP_DIMS)
    accept_score=dq + 0.035*removed - 0.050*added + 0.025*dim_improve - 0.040*dim_decline
    accepted=(
        candidate_audit["quality"] >= current_audit["quality"] + OP_MIN_ACCEPT_DELTA
        or (removed > added and candidate_audit["quality"] >= current_audit["quality"] - 0.025)
        or (dim_improve > dim_decline and candidate_audit["quality"] >= current_audit["quality"] - 0.025)
        or (not new_fail and candidate_audit["quality"] >= current_audit["quality"] - 0.050)
    )
    return accepted, accept_score, removed, added, dq, dim_improve, dim_decline

def recursive_operational_repair_contract_for_row(row, initial_contract, force=False):
    cache_dir=Path(OUTPUT_DIR)/"operational_repaired_contracts"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path=cache_dir/f"{row['id']}.json"
    raw_path=cache_dir/f"{row['id']}.raw.txt"
    trace_path=cache_dir/f"{row['id']}.trace.json"

    if cache_path.exists() and not force:
        try:
            loaded=normalize_contract(json.loads(cache_path.read_text(encoding="utf-8")), row_id=row["id"])
            trace=json.loads(trace_path.read_text(encoding="utf-8")) if trace_path.exists() else []
            return loaded, trace
        except Exception:
            pass

    current=normalize_contract(initial_contract,row_id=row["id"])
    initial_audit=operational_audit(row,current)
    current_audit=initial_audit
    best=current
    best_audit=current_audit
    trace=[]
    raw_logs=[]
    stall_count=0

    for depth in range(OP_MAX_RECURSION_DEPTH+1):
        axes=failure_axes_from_operational_audit(current_audit)
        trace_entry={
            "row_id":row["id"],
            "depth":depth,
            "quality":float(current_audit["quality"]),
            "F_need":float(current_audit["dims"]["need"]),
            "F_function":float(current_audit["dims"]["function"]),
            "F_boundary":float(current_audit["dims"]["boundary"]),
            "F_trap":float(current_audit["dims"]["trap"]),
            "F_collapse":float(current_audit["dims"]["collapse"]),
            "failures":list(current_audit.get("failures",[])),
            "axes":axes,
            "accepted":False,
            "delta_quality":0.0,
            "collapse_reason":"running",
            "family_class":current.get("family_class",""),
            "domain_carrier":" | ".join(current.get("domain_carrier",[])),
            "forbidden_neighbor_carrier":" | ".join(current.get("forbidden_neighbor_carrier",[])),
            "witness_readout":current.get("witness_readout",""),
        }
        trace.append(trace_entry)

        if not current_audit.get("failures",[]) and current_audit["quality"] >= OP_LOCK:
            trace[-1]["collapse_reason"]="psi_operational_lock"
            break
        if depth >= OP_MAX_RECURSION_DEPTH:
            trace[-1]["collapse_reason"]="omega_max_recursion_depth"
            break
        if stall_count >= OP_STALL_PATIENCE:
            trace[-1]["collapse_reason"]="omega_stall_no_operational_gain"
            break

        candidates=[]

        if ENABLE_LLM_OPERATIONAL_REPAIR:
            prompt=recursive_operational_repair_prompt(row,current,current_audit,trace,depth)
            raw=generate_text(prompt, max_new_tokens=OP_REPAIR_MAX_NEW_TOKENS, temperature=OP_REPAIR_TEMPERATURE)
            raw_logs.append(f"DEPTH {depth} LLM\n{raw}\n")
            obj=extract_json_object(raw)
            llm_candidate=normalize_contract(obj,row_id=row["id"])
            llm_audit=operational_audit(row,llm_candidate)
            candidates.append(("llm_operational_recursive",llm_candidate,llm_audit))

        scaffold=deterministic_operational_scaffold(row,current,current_audit)
        scaffold_audit=operational_audit(row,scaffold)
        candidates.append(("candidate_blind_operational_scaffold",scaffold,scaffold_audit))

        accepted_candidates=[]
        scored_candidates=[]

        for source,cand,cand_audit in candidates:
            accepted, accept_score, removed, added, dq, dim_improve, dim_decline=accept_operational_candidate(current_audit,cand_audit)
            scored_candidates.append((accept_score,source,cand,cand_audit,accepted,removed,added,dq,dim_improve,dim_decline))
            if accepted:
                accepted_candidates.append((accept_score,source,cand,cand_audit,removed,added,dq,dim_improve,dim_decline))

        scored_candidates=sorted(scored_candidates, key=lambda x: (x[0], x[3]["quality"]), reverse=True)

        if accepted_candidates:
            accept_score,source,next_contract,next_audit,removed,added,dq,dim_improve,dim_decline=sorted(
                accepted_candidates, key=lambda x: (x[0], x[3]["quality"]), reverse=True
            )[0]
            trace[-1].update({
                "accepted":True,
                "accepted_source":source,
                "accept_score":float(accept_score),
                "removed_failures":int(removed),
                "added_failures":int(added),
                "dim_improve":int(dim_improve),
                "dim_decline":int(dim_decline),
                "delta_quality":float(dq),
                "next_quality":float(next_audit["quality"]),
                "collapse_reason":"accepted_operational_recursive_step",
            })
            current=next_contract
            current_audit=next_audit
            if current_audit["quality"] > best_audit["quality"]:
                best=current
                best_audit=current_audit
            if dq < OP_MIN_ACCEPT_DELTA and removed <= added and dim_improve <= dim_decline:
                stall_count += 1
            else:
                stall_count=0
        else:
            stall_count += 1
            if scored_candidates:
                accept_score,source,cand,cand_audit,accepted,removed,added,dq,dim_improve,dim_decline=scored_candidates[0]
                trace[-1].update({
                    "accepted":False,
                    "accepted_source":source,
                    "accept_score":float(accept_score),
                    "removed_failures":int(removed),
                    "added_failures":int(added),
                    "dim_improve":int(dim_improve),
                    "dim_decline":int(dim_decline),
                    "delta_quality":float(dq),
                    "next_quality":float(cand_audit["quality"]),
                    "collapse_reason":"rejected_all_operational_candidates",
                })
            else:
                trace[-1]["collapse_reason"]="no_candidates_generated"

    final=normalize_contract(best,row_id=row["id"])
    final_audit=operational_audit(row,final)

    trace.append({
        "row_id":row["id"],
        "depth":"final",
        "quality":float(final_audit["quality"]),
        "F_need":float(final_audit["dims"]["need"]),
        "F_function":float(final_audit["dims"]["function"]),
        "F_boundary":float(final_audit["dims"]["boundary"]),
        "F_trap":float(final_audit["dims"]["trap"]),
        "F_collapse":float(final_audit["dims"]["collapse"]),
        "failures":list(final_audit.get("failures",[])),
        "axes":failure_axes_from_operational_audit(final_audit),
        "accepted":True,
        "delta_quality":float(final_audit["quality"]-initial_audit["quality"]),
        "collapse_reason":"final_best_operational_contract",
        "family_class":final.get("family_class",""),
        "domain_carrier":" | ".join(final.get("domain_carrier",[])),
        "forbidden_neighbor_carrier":" | ".join(final.get("forbidden_neighbor_carrier",[])),
        "witness_readout":final.get("witness_readout",""),
    })

    cache_path.write_text(json.dumps(final, indent=2), encoding="utf-8")
    raw_path.write_text("\n".join(raw_logs), encoding="utf-8")
    trace_path.write_text(json.dumps(trace, indent=2), encoding="utf-8")
    return final, trace

operational_contracts={}
all_trace_rows=[]
for row in tqdm(base_rows, desc="v68 operational recursive repair/load contracts"):
    c,trace=recursive_operational_repair_contract_for_row(row, contracts[row["id"]], force=REGENERATE_OPERATIONAL_REPAIRS)
    operational_contracts[row["id"]]=c
    all_trace_rows.extend(trace)

operational_audit_df=audit_df_for(operational_contracts,"operational_repaired")
operational_trace_df=pd.DataFrame(all_trace_rows)
audit_comparison_df=pd.concat([initial_audit_df, operational_audit_df], ignore_index=True)

display(operational_audit_df.sort_values("quality").head(24))
display(operational_trace_df.head(80))
print("initial mean operational quality:", initial_audit_df["quality"].mean())
print("repaired mean operational quality:", operational_audit_df["quality"].mean())

audit_comparison_df.to_csv(Path(OUTPUT_DIR)/"operational_audit_comparison.csv", index=False)
operational_audit_df.to_csv(Path(OUTPUT_DIR)/"operational_repaired_audit.csv", index=False)
operational_trace_df.to_csv(Path(OUTPUT_DIR)/"operational_repair_trace.csv", index=False)

pd.DataFrame([
    {
        "base_id":sid,
        "family_class":c.get("family_class",""),
        "domain_carrier":" | ".join(c.get("domain_carrier",[])),
        "forbidden_neighbor_carrier":" | ".join(c.get("forbidden_neighbor_carrier",[])),
        "boundary_conditions":" | ".join(c.get("boundary_conditions",[])),
        "preserved_function":c.get("preserved_function",""),
        "failure_modes":" | ".join(c.get("failure_modes",[])),
        "witness_readout":c.get("witness_readout",""),
        "residue":c.get("residue",None),
    }
    for sid,c in operational_contracts.items()
]).to_csv(Path(OUTPUT_DIR)/"operational_repaired_contracts.csv", index=False)


v68 operational recursive repair/load contracts:   0%|          | 0/24 [00:00<?, ?it/s]

,base_id,label,quality,F_need,F_function,F_boundary,F_trap,F_collapse,failures,family_class,domain_carrier,forbidden_neighbor_carrier,boundary_conditions,preserved_function,failure_modes,witness_readout
16,adv_ping_01,operational_repaired,0.597457,0.831244,0.563329,0.840683,0.787973,0.499761,collapse_failure | collapse_failure:placeholde...,operational closure of ping beacon shaped math...,ping | beacon | shaped | math | operationally ...,beacon | surface label without operational fit...,"{'preserve': 'shape', 'reject': 'non-beacon'}",being tested,colliding with non-ping shapes | not conformin...,"the chosen collapse preserves ping, beacon, sh..."
9,adv_breath_01,operational_repaired,0.600772,0.713159,0.625001,0.764216,0.827176,0.529184,collapse_failure | collapse_failure:placeholde...,operational closure of breathes non-moving mat...,non-moving | matter | changes,moving matter | surface label without operatio...,"{'preserve': 'without', 'reject': 'moving'}",functionality of non-moving matter preservation,collapse due to moving nature | failure in mai...,the chosen collapse preserves non-moving matte...
12,adv_tree_01,operational_repaired,0.610621,0.834422,0.611771,0.831773,0.837829,0.495221,collapse_failure | collapse_failure:placeholde...,operational closure of tree exposes leaf trunk...,exposes | tree | leaf | trunk | fruit | shade ...,bush | vine | surface label without operationa...,"{'preserve': ['leaf', 'trunk', 'fruit'], 'reje...",photosynthesis,rotting leaves | falling fruits | surface word...,"the chosen collapse preserves tree, exposes, l..."
19,adv_solution_01,operational_repaired,0.631411,0.758155,0.559654,0.789637,0.795776,0.558626,collapse_failure:placeholder_family_class,operational closure of solution merely string ...,lens | solution | merely | string,answer | string | surface label without operat...,"{'preserve': ['functionality'], 'reject': []}",remaining functionality,surface label match without operational fit | ...,"the chosen collapse preserves solution, merely..."
11,adv_flower_01,operational_repaired,0.668965,0.728250,0.822190,0.820869,0.763678,0.555549,collapse_failure:placeholder_family_class,operational closure of flower described hidden...,flower | hidden | rather | color,visible | bloom | describes | surface color,"{'preserve': ['hidden', 'rather'], 'reject': [...",remaining as a flower described hidden rather ...,disappearing or becoming invisible | surface w...,"the chosen collapse preserves flower, hidden, ..."
3,adv_house_01,operational_repaired,0.671642,0.685793,0.646301,0.879602,0.799774,0.552997,,structural,inward | house | presents | door | room | roof...,door | surface label without operational fit |...,"{'preserve': 'hidden inward fold', 'reject': '...",remaining structural integrity,collapse outward fold | rupture surface struct...,internal crease visible upon inspection
6,adv_sha_01,operational_repaired,0.673048,0.764649,0.623365,0.871322,0.761669,0.614172,collapse_failure:placeholder_family_class,operational closure of sha 256 produces digest...,digest | sha | 256 | produces | lens,general operation family,{'valid_digests': ['must have exactly 32 bytes...,hashing algorithm integrity preservation,collisions | preimage attacks | second preimag...,"the chosen collapse preserves sha, 256, produc..."
23,adv_commit_01,operational_repaired,0.675374,0.846385,0.743687,0.855394,0.664548,0.584628,collapse_failure:placeholder_family_class,operational closure of possible repair real un...,potential | commitment path | witness/readout,general operation family | operation family,preserve:commitment path | reject:surface labe...,remaining potential after repair,surface label match without operational fit,"the chosen collapse preserves possible, repair..."
7,adv_sha_02,operational_repaired,0.682304,0.670835,0.594497,0.745660,0.793495,0.739204,,rhyme,roles | functions | boundaries | traps | colla...,identical | non_rhyming_roles,identical_roles | too_thin_constraints,"rhyming roles, preserving functio

,row_id,depth,quality,F_need,F_function,F_boundary,F_trap,F_collapse,failures,axes,...,domain_carrier,forbidden_neighbor_carrier,witness_readout,accepted_source,accept_score,removed_failures,added_failures,dim_improve,dim_decline,next_quality
0,adv_coupler_01,0,0.685714,0.668946,0.757843,0.852928,0.658751,0.700505,[trap_failure:forbidden_too_thin],"[trap, forbidden_neighbor_carrier, failure_modes]",...,"rubber coupler, vacuum pump shaft",concrete | metal,observed radial compression without coupler mo...,candidate_blind_operational_scaffold,0.060673,1.0,0.0,1.0,1.0,0.726387
1,adv_coupler_01,1,0.726387,0.668946,0.697019,0.858375,0.799621,0.700505,[],[operational_densification],...,"rubber coupler, vacuum pump shaft",concrete | metal | surface label without opera...,observed radial compression without coupler mo...,candidate_blind_operational_scaffold,0.000000,0.0,0.0,0.0,0.0,0.726387
2,adv_coupler_01,2,0.726387,0.668946,0.697019,0.858375,0.799621,0.700505,[],[operational_densification],...,"rubber coupler, vacuum pump shaft",concrete | metal | surface label without opera...,observed radial compression without coupler mo...,candidate_blind_operational_scaffold,0.000000,0.0,0.0,0.0,0.0,0.726387
3,adv_coupler_01,3,0.726387,0.668946,0.697019,0.858375,0.799621,0.700505,[],[operational_densification],...,"rubber coupler, vacuum pump shaft",concrete | metal | surface label without opera...,observed radial compression without coupler mo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,adv_coupler_01,final,0.726387,0.668946,0.697019,0.858375,0.799621,0.700505,[],[operational_densification],...,"rubber coupler, vacuum pump shaft",concrete | metal | surface label without opera...,observed radial compression without coupler mo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,adv_surface_01,2,0.706177,0.707136,0.667227,0.850231,0.823421,0.621606,[],[operational_densification],...,socket | uses | word | repeatedly | fit | pres...,functionality | boundary | surface label witho...,the ability to detect changes in the shape of ...,candidate_blind_operational_scaffold,0.000000,0.0,0.0,0.0,0.0,0.706177
76,adv_surface_01,3,0.706177,0.707136,0.667227,0.850231,0.823421,0.621606,[],[operational_densification],...,socket | uses | word | repeatedly | fit | pres...,functionality | boundary | surface label witho...,the ability to detect changes in the shape of ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,adv_surface_01,final,0.706177,0.707136,0.667227,0.850231,0.823421,0.621606,[],[operational_densification],...,socket | uses | word | repeatedly | fit | pres...,functionality | boundary | surface label witho...,the ability to detect changes in the shape of ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,adv_surface_02,0,0.476994,0.439344,0.537458,0.870276,0.665048,0.644524,"[need_failure, function_failure, need_failure:...","[need, domain_carrier, function, preserved_fun...",...,a shape in space-time,Translation | Rotation,Witnesses can observe the change in volume as ...,candidate_blind_operational_scaffold,0.447797,4.0,0.0,3.0,0.0,0.709791


initial mean operational quality: 0.5438852230707805
repaired mean operational quality: 0.6930683632691702


In [10]:
# v68 generated contract scoring + decisions

def generated_contract_scores(row, contract):
    choices=row["choices"]
    pos_text=contract_to_text(contract, "positive")
    neg_text=contract_to_text(contract, "negative")
    domain_text=contract_to_text(contract, "domain")
    forbidden_text=contract_to_text(contract, "forbidden")
    witness_text=contract_to_text(contract, "witness")
    checklist_text=contract_to_text(contract, "full")
    full_text=contract_to_text(contract, "full")

    pos_cos=cos_scores(choices, pos_text)
    neg_cos=cos_scores(choices, neg_text) if neg_text.strip() else np.zeros(len(choices), dtype=np.float32)
    domain_cos=cos_scores(choices, domain_text) if domain_text.strip() else np.zeros(len(choices), dtype=np.float32)
    forbidden_cos=cos_scores(choices, forbidden_text) if forbidden_text.strip() else np.zeros(len(choices), dtype=np.float32)
    witness_cos=cos_scores(choices, witness_text) if witness_text.strip() else np.zeros(len(choices), dtype=np.float32)
    checklist_cos=cos_scores(choices, checklist_text) if checklist_text.strip() else np.zeros(len(choices), dtype=np.float32)
    full_cos=cos_scores(choices, full_text)

    pos_j=np.array([jaccard(c,pos_text) for c in choices], dtype=np.float32)
    neg_j=np.array([jaccard(c,neg_text) for c in choices], dtype=np.float32)
    domain_j=np.array([jaccard(c,domain_text) for c in choices], dtype=np.float32)
    forbidden_j=np.array([jaccard(c,forbidden_text) for c in choices], dtype=np.float32)
    witness_j=np.array([jaccard(c,witness_text) for c in choices], dtype=np.float32)
    checklist_j=np.array([jaccard(c,checklist_text) for c in choices], dtype=np.float32)
    full_j=np.array([jaccard(c,full_text) for c in choices], dtype=np.float32)

    pos=(0.65*zscore_local(pos_cos)+0.35*zscore_local(pos_j)).astype(np.float32)
    neg_avoid=(-0.65*zscore_local(neg_cos)-0.35*zscore_local(neg_j)).astype(np.float32)
    domain=(0.55*zscore_local(domain_cos)+0.45*zscore_local(domain_j)).astype(np.float32)
    forbidden=(0.55*zscore_local(forbidden_cos)+0.45*zscore_local(forbidden_j)).astype(np.float32)
    witness=(0.70*zscore_local(witness_cos)+0.30*zscore_local(witness_j)).astype(np.float32)
    checklist=(0.70*zscore_local(checklist_cos)+0.30*zscore_local(checklist_j)).astype(np.float32)
    full=(0.70*zscore_local(full_cos)+0.30*zscore_local(full_j)).astype(np.float32)

    table=pd.DataFrame({
        "choice_idx":range(len(choices)),
        "choice":choices,
        "choice_source_id":row.get("choice_source_ids",[""]*len(choices)),
        "pos":pos,
        "neg_avoid":neg_avoid,
        "domain":domain,
        "forbidden":forbidden,
        "witness":witness,
        "checklist":checklist,
        "full_contract":full,
    })
    return {"positive":pos,"negative_avoid":neg_avoid,"domain":domain,"forbidden":forbidden,"witness":witness,"checklist":checklist,"full_contract":full}, table

def generated_support(branches, pred):
    return (
        int(np.argmax(branches["positive"])==pred)
        + int(np.argmax(branches["negative_avoid"])==pred)
        + int(np.argmax(branches["domain"])==pred)
        + int(np.argmax(branches["witness"])==pred)
        + int(np.argmax(branches["checklist"])==pred)
        + int(np.argmax(branches["full_contract"])==pred)
    )

def compiled_baseline_decision(row, compiled_branches, base_scores, text_scores, support_min=1):
    raw=compiled_branches["full"]
    raw_pred,raw_margin=argmax_margin(raw)
    base_pred,_=argmax_margin(base_scores)
    text_pred,_=argmax_margin(text_scores)
    support=support_for_pred_from_branches(compiled_branches, raw_pred)
    evidence=(0.50*zscore_local(raw)+0.25*zscore_local(base_scores)+0.25*zscore_local(text_scores)).astype(np.float32)
    ev_pred,_=argmax_margin(evidence)
    if support>=support_min and (raw_pred==ev_pred or raw_pred==text_pred or raw_pred==base_pred):
        return raw_pred,"compiled_raw_safe",0,raw_margin,support,ev_pred
    return base_pred,"compiled_keep_base",1,raw_margin,support,ev_pred

def generated_decision(row, gen_branches, base_scores, text_scores, weights, support_min, margin_min, op_quality, op_min, base_protect_margin, override_support_min):
    wc, wd, wf, ww, wk = weights
    base_pred,base_margin=argmax_margin(base_scores)
    text_pred,_=argmax_margin(text_scores)

    contract_score=(wc*zscore_local(gen_branches["full_contract"])
                    + wd*zscore_local(gen_branches["domain"])
                    - wf*zscore_local(gen_branches["forbidden"])
                    + ww*zscore_local(gen_branches["witness"])
                    + wk*zscore_local(gen_branches["checklist"])
                    + 0.25*zscore_local(gen_branches["negative_avoid"])).astype(np.float32)

    pred,margin=argmax_margin(contract_score)
    support=generated_support(gen_branches, pred)
    evidence=(0.55*zscore_local(contract_score)+0.25*zscore_local(base_scores)+0.20*zscore_local(text_scores)).astype(np.float32)
    ev_pred,_=argmax_margin(evidence)

    safe=(op_quality>=op_min and support>=support_min and margin>=margin_min)
    agrees=(pred==ev_pred or pred==text_pred or pred==base_pred)
    base_protected=(base_margin>=base_protect_margin and pred!=base_pred and support<override_support_min)

    if safe and agrees and not base_protected:
        return pred,"generated_operational_contract_safe",0,margin,support,ev_pred,contract_score,evidence

    return base_pred,"omega_operational_contract_no_safe_collapse_keep_base",1,margin,support,ev_pred,contract_score,evidence


In [11]:
# v68 precompute row scores using operationally repaired contracts

precomp=[]
for row in tqdm(rows, desc="v68 precompute row scores"):
    prompt_text,base_scores,base_probs,base_pred,base_margin,base_ent=score_base(row)
    text_scores=score_answer_text_with_choices(row,prompt_text)
    compiled_branches=score_compiled_slot_branches(row, slot_by_base_id[row["base_id"]])
    gen_contract=operational_contracts[row["base_id"]]
    gen_audit=operational_audit(base_by_id[row["base_id"]], gen_contract)
    gen_branches, gen_table=generated_contract_scores(row, gen_contract)
    precomp.append({
        "row":row,
        "prompt_text":prompt_text,
        "base_scores":base_scores,
        "text_scores":text_scores,
        "compiled_branches":compiled_branches,
        "gen_contract":gen_contract,
        "gen_audit":gen_audit,
        "gen_branches":gen_branches,
        "gen_table":gen_table,
    })
print("precomputed:", len(precomp))


v68 precompute row scores:   0%|          | 0/96 [00:00<?, ?it/s]

precomputed: 96


KeyboardInterrupt: 

In [ ]:
# v68 final run

def final_run(config):
    weights=(float(config["wc_contract"]),float(config["wd_domain"]),float(config["wf_forbidden"]),float(config["ww_witness"]),float(config["wk_checklist"]))
    support_min=int(config["support_min"])
    margin_min=float(config["margin_min"])
    op_min=float(config["op_min"])
    base_protect_margin=float(config["base_protect_margin"])
    override_support_min=int(config["override_support_min"])

    rows_out=[]
    tables={}
    for pack in precomp:
        row=pack["row"]
        gold=row["answer_idx"]
        base_pred,base_margin=argmax_margin(pack["base_scores"])
        text_pred,text_margin=argmax_margin(pack["text_scores"])
        compiled_pred,compiled_reason,compiled_omega,compiled_margin,compiled_support,compiled_ev=compiled_baseline_decision(row,pack["compiled_branches"],pack["base_scores"],pack["text_scores"],support_min=1)
        gen_pred,gen_reason,gen_omega,gen_margin,gen_support,gen_ev,contract_score,evidence=generated_decision(
            row,pack["gen_branches"],pack["base_scores"],pack["text_scores"],
            weights=weights,support_min=support_min,margin_min=margin_min,
            op_quality=pack["gen_audit"]["quality"],op_min=op_min,
            base_protect_margin=base_protect_margin,override_support_min=override_support_min
        )
        tab=pack["gen_table"].copy()
        tab["base_score"]=pack["base_scores"]
        tab["text_score"]=pack["text_scores"]
        tab["generated_contract_score"]=contract_score
        tab["generated_evidence_score"]=evidence
        tables[row["id"]]=tab

        def pick(prefix,p):
            return {f"{prefix}_idx":p, f"{prefix}_choice":row["choices"][p], f"{prefix}_source_id":row["choice_source_ids"][p], f"{prefix}_correct":int(p==gold)}

        contract=pack["gen_contract"]
        audit=pack["gen_audit"]
        rec={
            "id":row["id"],
            "base_id":row["base_id"],
            "band":row["band"],
            "gold_idx":gold,
            "gold_choice":row["choices"][gold],
            "gold_source_id":row["choice_source_ids"][gold],
            "choice_source_ids":"|".join(row["choice_source_ids"]),
            "wc_contract":weights[0],"wd_domain":weights[1],"wf_forbidden":weights[2],"ww_witness":weights[3],"wk_checklist":weights[4],
            "support_min":support_min,"margin_min":margin_min,"op_min":op_min,"base_protect_margin":base_protect_margin,"override_support_min":override_support_min,
            "base_margin":base_margin,"text_margin":text_margin,"compiled_margin":compiled_margin,"compiled_support":compiled_support,
            "generated_margin":gen_margin,"generated_support":gen_support,
            "compiled_reason":compiled_reason,"generated_reason":gen_reason,"generated_omega":gen_omega,
            "op_quality":audit["quality"],
            "F_need":audit["dims"]["need"],"F_function":audit["dims"]["function"],"F_boundary":audit["dims"]["boundary"],"F_trap":audit["dims"]["trap"],"F_collapse":audit["dims"]["collapse"],
            "op_failures":" | ".join(audit["failures"]),
            "family_class":contract.get("family_class",""),
            "domain_carrier":" | ".join(contract.get("domain_carrier",[])),
            "forbidden_neighbor_carrier":" | ".join(contract.get("forbidden_neighbor_carrier",[])),
            "boundary_conditions":" | ".join(contract.get("boundary_conditions",[])),
            "preserved_function":contract.get("preserved_function",""),
            "failure_modes":" | ".join(contract.get("failure_modes",[])),
            "witness_readout":contract.get("witness_readout",""),
            "residue":contract.get("residue",None),
        }
        rec.update(pick("base",base_pred))
        rec.update(pick("text",text_pred))
        rec.update(pick("compiled",compiled_pred))
        rec.update(pick("generated",gen_pred))
        rows_out.append(rec)

    df=pd.DataFrame(rows_out)
    for mode in ["text","compiled","generated"]:
        df[f"{mode}_helped"]=((df.base_correct==0)&(df[f"{mode}_correct"]==1)).astype(int)
        df[f"{mode}_hurt"]=((df.base_correct==1)&(df[f"{mode}_correct"]==0)).astype(int)
    return df,tables

results_df, final_tables=final_run(best_config)

summary=pd.DataFrame([{
    "n":len(results_df),
    "n_base_items":results_df.base_id.nunique(),
    **{k:best_config[k] for k in ["wc_contract","wd_domain","wf_forbidden","ww_witness","wk_checklist","support_min","margin_min","op_min","base_protect_margin","override_support_min"]},
    "base_acc":results_df.base_correct.mean(),
    "text_acc":results_df.text_correct.mean(),
    "compiled_acc":results_df.compiled_correct.mean(),
    "generated_acc":results_df.generated_correct.mean(),
    "generated_gap_to_compiled":results_df.generated_correct.mean()-results_df.compiled_correct.mean(),
    "generated_gain_vs_base":results_df.generated_correct.mean()-results_df.base_correct.mean(),
    "compiled_helped":int(results_df.compiled_helped.sum()),
    "compiled_hurt":int(results_df.compiled_hurt.sum()),
    "generated_helped":int(results_df.generated_helped.sum()),
    "generated_hurt":int(results_df.generated_hurt.sum()),
    "generated_omega_count":int(results_df.generated_omega.sum()),
    "mean_op_quality":float(results_df.op_quality.mean()),
    "mean_F_need":float(results_df.F_need.mean()),
    "mean_F_function":float(results_df.F_function.mean()),
    "mean_F_boundary":float(results_df.F_boundary.mean()),
    "mean_F_trap":float(results_df.F_trap.mean()),
    "mean_F_collapse":float(results_df.F_collapse.mean()),
}])

by_band=results_df.groupby("band")[["base_correct","text_correct","compiled_correct","generated_correct","compiled_helped","compiled_hurt","generated_helped","generated_hurt","generated_omega","op_quality","F_need","F_function","F_boundary","F_trap","F_collapse"]].mean().reset_index()
by_base_item=results_df.groupby(["base_id","band","gold_source_id"])[["base_correct","text_correct","compiled_correct","generated_correct","compiled_helped","compiled_hurt","generated_helped","generated_hurt","generated_omega","op_quality","F_need","F_function","F_boundary","F_trap","F_collapse"]].mean().reset_index()
interesting=results_df[(results_df.generated_correct==0)|(results_df.compiled_correct != results_df.generated_correct)|(results_df.generated_idx != results_df.base_idx)|(results_df.generated_omega==1)].copy()

display(summary)
display(by_band)
display(by_base_item.sort_values("generated_correct").head(30))
display(interesting[["id","base_id","band","gold_choice","choice_source_ids","base_choice","base_correct","compiled_choice","compiled_correct","generated_choice","generated_correct","generated_reason","generated_omega","generated_margin","generated_support","op_quality","F_need","F_function","F_boundary","F_trap","F_collapse","op_failures","family_class","domain_carrier","forbidden_neighbor_carrier","witness_readout"]].head(160))


In [ ]:
# v68 save outputs + plots

out=Path(OUTPUT_DIR)
results_df.to_csv(out/"results.csv",index=False)
summary.to_csv(out/"summary.csv",index=False)
sweep_df.to_csv(out/"controller_sweep.csv",index=False)
ranked.to_csv(out/"controller_sweep_ranked.csv",index=False)
by_band.to_csv(out/"by_band.csv",index=False)
by_base_item.to_csv(out/"by_base_item.csv",index=False)
interesting.to_csv(out/"interesting_cases.csv",index=False)

for row_id,tab in final_tables.items():
    safe=row_id.replace("/","_")
    tab.to_csv(out/f"branches_{safe}.csv",index=False)

print("Saved v68 outputs in", out)
print("summary.csv, results.csv, operational_repair_trace.csv, operational_repaired_contracts.csv, controller_sweep_ranked.csv")

summary[["base_acc","text_acc","compiled_acc","generated_acc"]].T.plot(kind="bar",legend=False)
plt.title("v68 Recursive Operational Checklist Critic Accuracy")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

by_band.set_index("band")[["base_correct","compiled_correct","generated_correct"]].plot(kind="bar", figsize=(14,5))
plt.title("v68 Accuracy by Band")
plt.ylabel("accuracy")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

pd.DataFrame({
    "compiled":[results_df.compiled_helped.sum(),results_df.compiled_hurt.sum()],
    "generated":[results_df.generated_helped.sum(),results_df.generated_hurt.sum()],
}, index=["helped","hurt"]).plot(kind="bar")
plt.title("v68 Help vs Hurt")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

audit_compare = audit_comparison_df.pivot_table(index="base_id", columns="label", values="quality", aggfunc="mean")
audit_compare.plot(kind="bar", figsize=(16,5))
plt.title("Operational Contract Quality: Initial vs Repaired")
plt.ylabel("quality")
plt.xticks(rotation=60,ha="right")
plt.tight_layout()
plt.show()

dim_means=summary[["mean_F_need","mean_F_function","mean_F_boundary","mean_F_trap","mean_F_collapse"]].T
dim_means.plot(kind="bar", legend=False)
plt.title("Mean Five-Dimensional Operational Scores")
plt.ylabel("score")
plt.xticks(rotation=30,ha="right")
plt.tight_layout()
plt.show()

display(ranked[ranked.generated_hurt==0].head(50)[["wc_contract","wd_domain","wf_forbidden","ww_witness","wk_checklist","support_min","margin_min","op_min","base_protect_margin","override_support_min","generated_acc","generated_gap_to_compiled","generated_gain_vs_base","generated_helped","generated_hurt","generated_omega_count","compiled_acc"]])


In [ ]:
# v68 inspect hard cases and operational traces

display(operational_trace_df.tail(100))

for row_id in interesting["id"].head(30):
    print("="*100)
    print("CASE:", row_id)
    display(results_df[results_df.id==row_id][["id","base_id","band","gold_choice","choice_source_ids","base_choice","base_correct","compiled_choice","compiled_correct","generated_choice","generated_correct","generated_reason","generated_omega","op_quality","F_need","F_function","F_boundary","F_trap","F_collapse","op_failures","family_class","domain_carrier","forbidden_neighbor_carrier","boundary_conditions","preserved_function","failure_modes","witness_readout"]])
    display(final_tables[row_id].sort_values("generated_contract_score", ascending=False))
    base_id=results_df[results_df.id==row_id]["base_id"].iloc[0]
    print("OPERATIONAL TRACE:")
    display(operational_trace_df[operational_trace_df.row_id==base_id])


## v68 Readout

The clean proof files are:

```text
v68_outputs_.../summary.csv
v68_outputs_.../results.csv
v68_outputs_.../operational_repair_trace.csv
v68_outputs_.../operational_repaired_contracts.csv
```

Interpretation:

### Ψ-safe constructor

$$
\text{generated\_hurt}=0
$$

and:

$$
\text{generated\_acc}>\text{base\_acc}
$$

The operational critic is safe and useful.

### Ψ-strong constructor

$$
\text{generated\_hurt}=0
$$

and:

$$
\text{generated\_acc}\ge \text{compiled\_acc}-\epsilon
$$

The generated operational contract nearly matches the compiled-slot baseline.

### Ω unsafe constructor

$$
\text{generated\_hurt}>0
$$

The constructor remains unsafe.

Next fold:

$$
(Q,C_0,A_0,\Omega_0,C_T,\Psi/\Omega)
\rightarrow
\text{slot-constructor fine-tuning set}
$$

The training target is not the answer.

The training target is the operational contract that makes the answer lawful.
